# ViKAG-Faith, đợt 2: bổ sung theo đề cương

1. **Runtime → Change runtime type → T4 GPU**, rồi **Runtime → Run all**. Cho phép kết nối Google Drive.
2. Tổng thời gian ước tính khoảng 2 giờ, chia thành 5 phần theo thứ tự ưu tiên. Không cần theo dõi liên tục.
3. Nếu phiên bị ngắt, chỉ cần **Run all** lại: phần đã xong sẽ tự bỏ qua. Nếu hết thời gian, phần đã xong vẫn dùng được.
4. Cuối cùng tải `MyDrive/vikag-faith/vikag_dot2.zip` về.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → T4 GPU"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Mã nguồn (sinh tự động từ `src/`)

In [ ]:
!mkdir -p /content/vikag-faith/src

In [ ]:
%%writefile /content/vikag-faith/src/common.py
"""Shared paths, data loading, and Ollama helpers for the ViKAG-Faith pilot."""

import json
import os
import random
import re
import sys
import time
from pathlib import Path

import numpy as np
import ollama

ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "data"
OUT = ROOT / "outputs"
DATASET = ROOT / "external" / "ViHERMES" / "dataset" / "dataset.jsonl"
GRAPH_DB = DATA / "graph.db"
DENSE_INDEX = DATA / "dense_bge-m3.npz"

GEN_MODEL = "qwen2.5:3b"
BIG_MODEL = "qwen2.5:7b"
EMBED_MODEL = "bge-m3"
SEED = 42
# The local GTX 1650 hangs the whole machine under sustained load (driver TDR -> hard freeze),
# so models run on the CPU unless VIKAG_CPU=0 (set on Colab, which has a T4).
USE_CPU = os.environ.get("VIKAG_CPU", "1") == "1"
CPU_ONLY = {"num_gpu": 0} if USE_CPU else {}

sys.stdout.reconfigure(encoding="utf-8")
OUT.mkdir(exist_ok=True)


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def append_jsonl(path, row):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_dataset():
    rows = read_jsonl(DATASET)
    for i, r in enumerate(rows):
        r["row"] = i
    return rows


def sample_questions(per_hop=20):
    """Stratified sample: `per_hop` questions for each hop level 1-5, fixed seed."""
    rows = load_dataset()
    rng = random.Random(SEED)
    picked = []
    for hop in range(1, 6):
        pool = [r for r in rows if r["hop_level"] == hop]
        picked += rng.sample(pool, per_hop)
    return picked


class OllamaEmbedder:
    """Implements the ViHERMES `Embedder` port with bge-m3 served by Ollama."""

    identity = "ollama:" + EMBED_MODEL

    def __init__(self):
        self.cache = {}

    def encode(self, texts, batch=8):
        if texts and all(t in self.cache for t in texts):
            return np.asarray([self.cache[t] for t in texts], dtype=np.float32)
        out = []
        for i in range(0, len(texts), batch):
            chunk = texts[i : i + batch]
            try:
                out += ollama.embed(model=EMBED_MODEL, input=chunk, options=CPU_ONLY)["embeddings"]
            except Exception as e:  # runner crashes on some batches: fall back to one at a time
                print("embed batch failed, retrying singly:", str(e)[:80])
                time.sleep(3)
                out += [ollama.embed(model=EMBED_MODEL, input=[t], options=CPU_ONLY)["embeddings"][0]
                        for t in chunk]
            if len(texts) > batch:
                print(f"embedded {len(out)}/{len(texts)}", flush=True)
        self.cache.update(zip(texts, out))
        return np.asarray(out, dtype=np.float32)

    def unload(self):
        """Free the 4GB GPU before an LLM is loaded: both together crash the GTX 1650."""
        ollama.embed(model=EMBED_MODEL, input=["."], keep_alive=0, options=CPU_ONLY)


def chat(prompt, system=None, model=GEN_MODEL, fmt=None, num_predict=512, num_ctx=6144, temperature=0, seed=SEED):
    messages = ([{"role": "system", "content": system}] if system else []) + [
        {"role": "user", "content": prompt}
    ]
    options = {"temperature": temperature, "seed": seed, "num_predict": num_predict, "num_ctx": num_ctx, **CPU_ONLY}
    for attempt in range(3):
        try:
            r = ollama.chat(model=model, messages=messages, format=fmt, options=options)
            return r["message"]["content"]
        except ollama.ResponseError as e:
            if "repeat" in str(e):
                # Newer Ollama aborts degenerate loops; retry less greedily, then give up on this call.
                if attempt == 2:
                    print("ollama gave up (repetition):", str(e)[:80])
                    return ""
                options = {**options, "temperature": 0.3, "repeat_penalty": 1.15 + 0.1 * attempt}
                continue
            if attempt == 2:
                raise
            print("ollama retry:", e)
            time.sleep(3)
        except Exception as e:  # transient server errors: retry, then surface
            if attempt == 2:
                raise
            print("ollama retry:", e)
            time.sleep(3)


def chat_json(prompt, system=None, model=GEN_MODEL, **kw):
    """Ask for JSON output; returns {} when the model's output cannot be parsed."""
    text = chat(prompt, system, model, fmt="json", **kw)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.S)
        try:
            return json.loads(m.group()) if m else {}
        except json.JSONDecodeError:
            return {}

In [ ]:
%%writefile /content/vikag-faith/src/rules.py
"""Deterministic number / date / document-code check for claims against evidence."""

import re
import unicodedata

DOC_CODE = re.compile(r"\b\d{1,4}/\d{4}/[A-ZĐ][A-ZĐ0-9-]*", re.I)
DATE_WORDS = re.compile(r"ngày\s+(\d{1,2})\s+tháng\s+(\d{1,2})\s+năm\s+(\d{4})", re.I)
DATE_NUM = re.compile(r"\b(\d{1,2})[/.-](\d{1,2})[/.-](\d{4})\b")
NUMBER = re.compile(r"\b\d+(?:[.,]\d+)*\s*%?")
WORD = re.compile(r"\w+")


def norm(text):
    return unicodedata.normalize("NFC", text)


def values(text):
    """Typed values in a text: ('code', ...), ('date', d/m/y), ('num', ...)."""
    text = norm(text)
    out = set()
    for m in DOC_CODE.finditer(text):
        out.add(("code", m.group().upper()))
    text = DOC_CODE.sub(" ", text)
    for rx in (DATE_WORDS, DATE_NUM):
        for d, mth, y in rx.findall(text):
            out.add(("date", f"{int(d)}/{int(mth)}/{y}"))
        text = rx.sub(" ", text)
    for m in NUMBER.finditer(text):
        v = m.group().replace(" ", "").replace(",", ".").rstrip(".")
        if v:
            out.add(("num", v.lstrip("0") or "0"))
    return out


def sentences(text):
    return [s for s in re.split(r"(?<=[.;:!?])\s+|\n+", norm(text)) if s.strip()]


def overlap(a, b):
    wa = {w for w in WORD.findall(a.lower()) if not w.isdigit()}
    wb = {w for w in WORD.findall(b.lower()) if not w.isdigit()}
    return len(wa & wb) / max(1, len(wa))


def check(claim, evidence_texts):
    """None if every value in the claim occurs in the evidence; otherwise 'contradicted' when a
    closely matching evidence sentence carries a competing value of the same type (one the claim
    does not mention), else 'unsupported'."""
    claimed = values(claim)
    if not claimed:
        return None
    pool = set()
    for t in evidence_texts:
        pool |= values(t)
    missing = claimed - pool
    if not missing:
        return None
    kinds = {k for k, _ in missing}
    for t in evidence_texts:
        for s in sentences(t):
            competing = {k for k, v in values(s) - claimed}
            if overlap(claim, s) >= 0.5 and kinds & competing:
                return "contradicted"
    return "unsupported"

In [ ]:
%%writefile /content/vikag-faith/src/s10_dot2.py
"""Second-round analysis: new verifier variants, the 14B judge and the small Qwen verifier with spans.

Same test split, thresholds tuned on the same dev split with the same grid as s7. Writes
outputs/analysis/results_dot2.json and leaves results.json untouched, so first-round numbers do not move.
Silver labels stay the reference on the test split; the human column uses the same 165 blind labels.
"""

import json
import os
from collections import defaultdict

import numpy as np
from scipy.stats import kendalltau, spearmanr
from sklearn.metrics import f1_score, roc_auc_score

from common import OUT, SEED, read_jsonl

ANN = OUT / "annotation"
RES = OUT / "analysis"
LABELS = ["supported", "contradicted", "unsupported"]
GRID = np.round(np.arange(0.1, 0.95, 0.05), 2)
SCORE_FILE = os.environ.get("VIKAG_DOT2_SCORES", "verifier_scores_dot2.jsonl")
JUDGE_FILE = os.environ.get("VIKAG_DOT2_JUDGES", "judges_dot2.jsonl")
SPAN_FILE = os.environ.get("VIKAG_DOT2_SPAN", "judges_span.jsonl")


def macro_f1(y, p):
    return float(f1_score(y, p, labels=LABELS, average="macro", zero_division=0))


def decide(entail, contra, ts, tc):
    return "supported" if entail >= ts else "contradicted" if contra >= tc else "unsupported"


def human_labels():
    ids = json.load(open(ANN / "id_map.json", encoding="utf-8"))
    out = {}
    for name in ("C", "G", "H", "I", "J"):
        raw = json.load(open(ANN / f"vifaith_labels_{name}.json", encoding="utf-8"))["labels"]
        out.update({ids[k]: v["label"] for k, v in raw.items() if v.get("label") and ids.get(k)})
    return out


def table_row(p, keys, gold, human, score=None):
    ks = [k for k in keys["test"] if k in p]
    y, yp = [gold[k] for k in ks], [p[k] for k in ks]
    rng = np.random.default_rng(SEED)
    boots = [macro_f1([y[i] for i in idx], [yp[i] for i in idx]) for idx in (rng.integers(0, len(ks), len(ks)) for _ in range(1000))]
    wrong = [k for k in ks if gold[k] != "supported"]
    right = [k for k in ks if gold[k] == "supported"]
    hs = [k for k in keys["usable"] if k in human and k in p]
    yb = [int(v != "supported") for v in y]
    sb = [1 - score[k] for k in ks] if score else [int(v != "supported") for v in yp]
    return {"n": len(ks), "macro_f1": macro_f1(y, yp), "ci95": [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))],
            "auroc_not_supported": float(roc_auc_score(yb, sb)), "caught": sum(p[k] != "supported" for k in wrong), "wrong": len(wrong),
            "false_alarm": sum(p[k] != "supported" for k in right), "right": len(right),
            "n_human": len(hs), "macro_f1_human": macro_f1([human[k] for k in hs], [p[k] for k in hs])}


def paired(a, b, keys, gold, labels=None):
    ref = labels or gold
    pool = keys["usable"] if labels else keys["test"]
    ks = [k for k in pool if k in a and k in b and k in ref]
    y, pa, pb = np.array([ref[k] for k in ks]), np.array([a[k] for k in ks]), np.array([b[k] for k in ks])
    rng = np.random.default_rng(SEED)
    d = [macro_f1(y[i], pa[i]) - macro_f1(y[i], pb[i]) for i in (rng.integers(0, len(ks), len(ks)) for _ in range(1000))]
    return {"n": len(ks), "diff": macro_f1(y, pa) - macro_f1(y, pb), "ci95": [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
            "p_le_0": float(np.mean(np.array(d) <= 0))}


def main():
    res = json.load(open(RES / "results.json", encoding="utf-8"))
    pre = json.load(open(OUT / "prelabels.json", encoding="utf-8"))
    sample = {r["qid"]: r["split"] for r in read_jsonl(OUT / "sample.jsonl")}
    claims = {c["cid"]: c for c in read_jsonl(OUT / "claims.jsonl") if c["qid"] in sample}
    usable = [k for k in claims if k in pre and not pre[k]["bad"]]
    gold = {k: pre[k]["label"] for k in usable}
    keys = {"usable": usable, "dev": [k for k in usable if sample[claims[k]["qid"]] == "dev"],
            "test": [k for k in usable if sample[claims[k]["qid"]] == "test"]}
    human = human_labels()

    def tune(rows, name):
        y = [gold[k] for k in keys["dev"]]
        best = max(((a, b) for a in GRID for b in GRID),
                   key=lambda t: macro_f1(y, [decide(rows[k][name]["entail"], rows[k][name]["contra"], *t) for k in keys["dev"]]))
        return [float(x) for x in best]

    first = {r["cid"]: r for r in read_jsonl(OUT / "verifier_scores.jsonl")}
    th = res["thresholds"]
    base = {n: {k: decide(first[k][n]["entail"], first[k][n]["contra"], *th[n]) for k in usable} for n in ("zs_v1", "ft_v1")}
    judges7 = {}
    judges = read_jsonl(OUT / "judges.jsonl")
    for j in judges:
        if j["model"] == "qwen2.5:7b":
            judges7.update({k: v for k, v in j["judge"].items() if v and k in gold})

    out = {"methods": {}, "thresholds": {}, "paired_silver": {}, "paired_human": {}}
    new = {}
    path = OUT / SCORE_FILE
    if path.exists():
        rows = {r["cid"]: r for r in read_jsonl(path)}
        names = sorted({n for r in rows.values() for n in r if n != "cid"})
        for n in names:
            if not all(n in rows.get(k, {}) for k in usable):
                print("skipping", n, "- scores missing")
                continue
            t = tune(rows, n)
            out["thresholds"][n] = t
            new["nli_" + n] = {k: decide(rows[k][n]["entail"], rows[k][n]["contra"], *t) for k in usable}
            out["methods"]["nli_" + n] = table_row(new["nli_" + n], keys, gold, human, {k: rows[k][n]["entail"] for k in usable})

    # answer-level reference: silver GGF per answer on the test split, as in s7
    by_answer = defaultdict(list)
    for k in keys["test"]:
        by_answer[(claims[k]["qid"], claims[k]["system"])].append(k)
    ggf = {a: sum(gold[k] == "supported" for k in ks) / len(ks) for a, ks in by_answer.items()}

    def corr(score):
        ks = [a for a in ggf if score.get(a) is not None]
        rho = spearmanr([ggf[a] for a in ks], [score[a] for a in ks]).statistic
        tau = kendalltau([ggf[a] for a in ks], [score[a] for a in ks]).statistic
        return {"n": len(ks), "spearman": float(rho), "kendall": float(tau)}

    path = OUT / JUDGE_FILE
    if path.exists():
        rows = read_jsonl(path)
        p = {k: v for j in rows for k, v in j["judge"].items() if v and k in gold}
        new["judge_14b"] = p
        out["methods"]["judge_14b"] = table_row(p, keys, gold, human)
        out["judge_coverage_14b"] = len(p)
        ragas = {(j["qid"], j["system"]): j["ragas"]["score"] for j in rows}
        out["corr_ragas_14b"] = corr(ragas)
        pa = {(j["qid"], j["system"]): [v == "supported" for k, v in j["judge"].items() if v and k in gold] for j in rows}
        out["corr_judge_14b"] = corr({a: sum(v) / len(v) for a, v in pa.items() if v})

    path = OUT / SPAN_FILE
    if path.exists():
        pools = {(p["qid"], p["system"]): " ".join(x["text"] for x in p["pool"]) for p in read_jsonl(OUT / "pools.jsonl")}
        rows = read_jsonl(path)
        p, found, given = {}, 0, 0
        for j in rows:
            text = " ".join(pools[(j["qid"], j["system"])].split())
            for k, v in j["judge"].items():
                if not v or k not in gold:
                    continue
                p[k] = v["label"]
                span = " ".join(v.get("span", "").split())
                if span:
                    given += 1
                    found += span in text
        new["qwen3b_span"] = p
        out["methods"]["qwen3b_span"] = table_row(p, keys, gold, human)
        out["span"] = {"given": given, "verbatim": found, "rate": found / max(1, given)}

    for name, pred in new.items():
        for ref_name, ref in (("nli_ft_v1", base["ft_v1"]), ("nli_zs_v1", base["zs_v1"]), ("judge_7b", judges7)):
            out["paired_silver"][f"{name} - {ref_name}"] = paired(pred, ref, keys, gold)
            out["paired_human"][f"{name} - {ref_name}"] = paired(pred, ref, keys, gold, human)
    json.dump(out, open(RES / "results_dot2.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("method                 macroF1   CI95            AUROC  caught  falseAl  human")
    for n, r in out["methods"].items():
        print(f"{n:22s} {r['macro_f1']:.3f}  {r['ci95'][0]:.2f}-{r['ci95'][1]:.2f}  {r['auroc_not_supported']:.3f}  "
              f"{r['caught']}/{r['wrong']}  {r['false_alarm']}/{r['right']}  {r['macro_f1_human']:.3f} ({r['n_human']})")
    for k in ("corr_ragas_14b", "corr_judge_14b", "span", "judge_coverage_14b"):
        if k in out:
            print(k, out[k])
    print("baseline: judge_7b", round(res["claim_level"]["judge_7b"]["macro_f1"], 3), "ft_v1", round(res["claim_level"]["nli_ft_v1"]["macro_f1"], 3))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vikag-faith/src/s1_answer.py
"""Step 1: answer the sampled ViHERMES questions with two systems sharing one LLM.

naive  - hybrid BM25 + bge-m3 seeds, cut at a character budget
graph  - same seeds, then ViHERMES typed propagation over the regulatory graph
Both get the same generator, prompt and context budget, so differences come from retrieval.
"""

from vihermes.graph import SQLiteGraph
from vihermes.retrieval import GraphPropagator, HybridRetriever, NumpyDenseIndex, units

from common import (
    DENSE_INDEX, GEN_MODEL, GRAPH_DB, OUT, OllamaEmbedder, append_jsonl, chat, read_jsonl,
    sample_questions,
)

BUDGET = 6000  # characters of passage text per question, both systems
OUTFILE = OUT / "answers.jsonl"

SYSTEM = (
    "Bạn là trợ lý hỏi đáp về văn bản quy phạm pháp luật ngành y tế Việt Nam. "
    "Chỉ dùng thông tin trong NGỮ CẢNH. Trả lời bằng tiếng Việt, 2-5 câu, nêu tên văn bản khi có. "
    "Nếu ngữ cảnh không đủ, nói rõ là không tìm thấy thông tin."
)


def passage(node):
    return {"id": node.id, "title": node.title, "text": node.text}


def take_budget(nodes):
    out, used = [], 0
    for n in nodes:
        if used + len(n.text) > BUDGET and out:
            break
        out.append(n)
        used += len(n.text)
    return out


def format_context(passages, relations=()):
    parts = [f"[{i}] Văn bản: {p['title']}\n{p['text']}" for i, p in enumerate(passages, 1)]
    if relations:
        parts.append("Quan hệ giữa các văn bản:\n" + "\n".join(relations))
    return "\n\n".join(parts)


def describe(edge, graph):
    names = {"AMENDS": "được sửa đổi bởi", "REPLACES": "được thay thế bởi",
             "SUPPLEMENTS": "được bổ sung bởi", "REFERS_TO": "dẫn chiếu tới"}
    if edge.relation not in names:
        return None
    s, t = graph.get(edge.source), graph.get(edge.target)
    label = lambda n, key: (n.title if n and n.title else key)  # noqa: E731
    return f"- {label(s, edge.source)} {names[edge.relation]} {label(t, edge.target)}"


def main():
    graph = SQLiteGraph(GRAPH_DB)
    corpus = units(graph)
    embedder = OllamaEmbedder()
    dense = NumpyDenseIndex(corpus, embedder, DENSE_INDEX)
    retriever = HybridRetriever(graph, dense, weight=0.5)
    propagator = GraphPropagator(graph, max_depth=4, max_nodes=8, max_chars=BUDGET)

    done = {(r["qid"], r["system"]) for r in read_jsonl(OUTFILE)} if OUTFILE.exists() else set()
    questions = sample_questions()
    embedder.encode([q["question"] for q in questions])  # cache, then free the GPU for the LLM
    embedder.unload()
    for qi, q in enumerate(questions):
        seeds = retriever.seeds(q["question"], k=8)
        runs = {"naive": (take_budget([e.node for e in seeds]), [])}
        expanded = propagator.expand(seeds[:4])
        gnodes = [e.node for e in expanded.evidence if e.node.kind != "Document" and e.node.text]
        rels = [d for d in (describe(e, graph) for e in expanded.edges) if d]
        runs["graph"] = (take_budget(gnodes), rels[:12])
        for system, (nodes, relations) in runs.items():
            if (q["id"] + f"#{q['row']}", system) in done:
                continue
            passages = [passage(n) for n in nodes]
            prompt = f"NGỮ CẢNH:\n{format_context(passages, relations)}\n\nCÂU HỎI: {q['question']}"
            answer = chat(prompt, SYSTEM, GEN_MODEL, num_predict=400, num_ctx=4096)
            append_jsonl(OUTFILE, {
                "qid": q["id"] + f"#{q['row']}", "system": system, "hop": q["hop_level"],
                "question": q["question"], "gold_answer": q["answer"],
                "gold_ids": q["used_context_ids"], "answer": answer,
                "retrieved": passages, "relations": relations,
            })
        print(f"{qi + 1}/{len(questions)} hop={q['hop_level']}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vikag-faith/src/s2_claims.py
"""Step 2: split every system answer into atomic, self-contained claims (Vietnamese prompt)."""

import re

from common import GEN_MODEL, OUT, append_jsonl, chat_json, read_jsonl, write_jsonl

SYSTEM = (
    "Bạn tách câu trả lời thành các mệnh đề nguyên tử (claim). Quy tắc:\n"
    "1. Mỗi claim chỉ nêu MỘT sự kiện, có một vị từ chính.\n"
    "2. Mỗi claim phải tự hiểu được: thay đại từ ('văn bản này', 'nó') bằng tên cụ thể.\n"
    "3. Giữ nguyên con số, ngày tháng, số hiệu văn bản như trong câu trả lời.\n"
    "4. Không thêm thông tin ngoài câu trả lời. Bỏ các câu chỉ nói 'không tìm thấy thông tin'.\n"
    'Trả về JSON: {"claims": ["...", "..."]}'
)

FEWSHOT = """Ví dụ:
CÂU HỎI: Thông tư 01/2020/TT-BYT có hiệu lực khi nào và quy định gì?
CÂU TRẢ LỜI: Thông tư 01/2020/TT-BYT có hiệu lực từ ngày 01/3/2020. Văn bản này quy định hồ sơ đăng ký lưu hành thuốc gồm đơn đăng ký và giấy chứng nhận GMP.
JSON: {"claims": ["Thông tư 01/2020/TT-BYT có hiệu lực từ ngày 01/3/2020.", "Thông tư 01/2020/TT-BYT quy định hồ sơ đăng ký lưu hành thuốc gồm đơn đăng ký.", "Thông tư 01/2020/TT-BYT quy định hồ sơ đăng ký lưu hành thuốc gồm giấy chứng nhận GMP."]}

CÂU HỎI: Ai chịu trách nhiệm tổ chức thực hiện chương trình?
CÂU TRẢ LỜI: Ngữ cảnh không nêu rõ. Tuy nhiên, Bộ Y tế chủ trì và phối hợp với các bộ, ngành liên quan.
JSON: {"claims": ["Bộ Y tế chủ trì tổ chức thực hiện chương trình.", "Bộ Y tế phối hợp với các bộ, ngành liên quan để thực hiện chương trình."]}
"""

ABSTAIN = re.compile(r"không (tìm thấy|có|đủ) thông tin|không được (nêu|đề cập)", re.I)
NUMBER = re.compile(r"\d+(?:[.,/]\d+)*")


def clean(claims):
    seen, out = set(), []
    for c in claims:
        if not isinstance(c, str):
            continue
        c = " ".join(c.split())
        if len(c) < 8 or c.lower() in seen or ABSTAIN.search(c):
            continue
        seen.add(c.lower())
        out.append(c)
    return out


def main():
    answers = read_jsonl(OUT / "answers.jsonl")
    # Raw splits are cached per answer so an interrupted run resumes.
    raw_path = OUT / "claims_raw.jsonl"
    raw = {(r["qid"], r["system"]): r["claims"] for r in read_jsonl(raw_path)} if raw_path.exists() else {}
    rows = []
    for i, a in enumerate(answers):
        key = (a["qid"], a["system"])
        if key not in raw:
            prompt = f"{FEWSHOT}\nCÂU HỎI: {a['question']}\nCÂU TRẢ LỜI: {a['answer']}\nJSON:"
            raw[key] = chat_json(prompt, SYSTEM, GEN_MODEL, num_predict=600).get("claims", [])
            append_jsonl(raw_path, {"qid": a["qid"], "system": a["system"], "claims": raw[key]})
        claims = clean(raw[key])
        for j, c in enumerate(claims):
            rows.append({
                "cid": f"{a['qid']}|{a['system']}|{j}", "qid": a["qid"], "system": a["system"],
                "hop": a["hop"], "claim": c,
                # Rule post-check: claims carrying several values are flagged for annotators.
                "multi_value": len(NUMBER.findall(c)) >= 3,
            })
        if not claims:
            print("no claims:", a["qid"], a["system"])
        print(f"{i + 1}/{len(answers)}", flush=True)
    write_jsonl(OUT / "claims.jsonl", rows)
    print("claims:", len(rows))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vikag-faith/src/s3_pool.py
"""Step 3: build each answer's evidence pool and the graph facts attached to it,
then draw the paired annotation sample (40 questions x 2 systems).

pool        = passages the system retrieved + the question's gold supporting passages
graph facts = provenance (passage -> document title) + legal relations from the ViHERMES graph
"""

import csv
import random

from vihermes.graph import SQLiteGraph

from common import DATA, GRAPH_DB, OUT, SEED, read_jsonl, write_jsonl

NAMES = {"AMENDS": "được sửa đổi bởi", "REPLACES": "được thay thế bởi",
         "SUPPLEMENTS": "được bổ sung bởi", "REFERS_TO": "dẫn chiếu tới"}


def label(graph, key):
    n = graph.get(key)
    if n is None:
        return key
    if n.title:
        return n.title
    return f"{n.document_id} (chưa có văn bản trong corpus)" if n.placeholder else key


def graph_facts(graph, pid):
    """Facts about one passage that come from the graph, not from its body text."""
    node = graph.get(pid)
    facts = []
    if node is not None:
        doc = graph.get(node.document_id)
        facts.append(f"Đoạn thuộc văn bản: {doc.title if doc and doc.title else node.title}")
        if not node.document_id.startswith("doc-"):
            facts.append(f"Số hiệu văn bản: {node.document_id}")
    for e in graph.edges(pid) + graph.edges(pid, incoming=True):
        if e.relation in NAMES:
            facts.append(f"{label(graph, e.source)} {NAMES[e.relation]} {label(graph, e.target)}")
    return list(dict.fromkeys(facts))


def main():
    graph = SQLiteGraph(GRAPH_DB)
    contexts = {r["context_id"]: r for r in csv.DictReader(open(DATA / "contexts.csv", encoding="utf-8"))}
    answers = read_jsonl(OUT / "answers.jsonl")
    claims = read_jsonl(OUT / "claims.jsonl")
    with_claims = {(c["qid"], c["system"]) for c in claims}

    pools = []
    for a in answers:
        ids = list(dict.fromkeys([p["id"] for p in a["retrieved"]] + a["gold_ids"]))
        pool = []
        for pid in ids:
            ctx = contexts.get(pid)
            if ctx is None:  # retrieved non-context nodes do not occur for context imports
                continue
            pool.append({
                "id": pid, "title": ctx["title"], "text": ctx["text"],
                "retrieved": pid in {p["id"] for p in a["retrieved"]},
                "gold": pid in a["gold_ids"], "facts": graph_facts(graph, pid),
            })
        pools.append({"qid": a["qid"], "system": a["system"], "hop": a["hop"], "pool": pool})
    write_jsonl(OUT / "pools.jsonl", pools)

    # Paired sample: questions where both systems produced claims, 8 per hop.
    rng = random.Random(SEED)
    sample = []
    for hop in range(1, 6):
        qids = sorted({a["qid"] for a in answers if a["hop"] == hop
                       and (a["qid"], "naive") in with_claims and (a["qid"], "graph") in with_claims})
        sample += rng.sample(qids, min(8, len(qids)))
    rng.shuffle(sample)
    # 20% of questions -> dev (threshold tuning), rest -> test.
    split = {q: ("dev" if i < len(sample) // 5 else "test") for i, q in enumerate(sample)}
    write_jsonl(OUT / "sample.jsonl", [{"qid": q, "split": split[q]} for q in sample])
    n = sum(1 for c in claims if c["qid"] in split)
    print(f"pools: {len(pools)}  sampled questions: {len(sample)}  sampled claims: {n}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vikag-faith/src/s3b_triples.py
"""Step 3b: triple store and mutual index between triples and source passages (proposal, evidence alignment).

triple        = (document or clause; legal relation; document) taken from the ViHERMES graph
chunk_triples = passage id -> triples of its document and of the passage itself
doc_chunks    = document number -> its passages, so a document named in a claim leads back to its source text

Writes outputs/triple_index.json. Uses no model.
"""

import csv
import json
import re
import sqlite3

from common import DATA, GRAPH_DB, OUT

DOC_NO = re.compile(r"\d+/(?:\d{4}/)?[A-Za-zĐđ]+(?:-[A-Za-zĐđ]+)*")
NAMES = {"AMENDS": "sửa đổi", "REPLACES": "thay thế", "SUPPLEMENTS": "bổ sung", "REFERS_TO": "dẫn chiếu tới"}
csv.field_size_limit(10 ** 8)


def main():
    db = sqlite3.connect(GRAPH_DB)
    nodes = {i: json.loads(p) for i, p in db.execute("select id, payload from nodes")}

    def name(key):
        n = nodes.get(key) or {}
        if n.get("title"):
            return n["title"]
        return n.get("document_id") or key

    def doc_of(key):
        return (nodes.get(key) or {}).get("document_id") or key

    by_doc, by_node, total = {}, {}, 0
    for src, tgt, rel in db.execute("select source, target, relation from edges"):
        if rel not in NAMES:
            continue
        t = f"({name(src)}; {NAMES[rel]}; {name(tgt)})."
        total += 1
        for key in (src, tgt):
            by_node.setdefault(key, []).append(t)
            by_doc.setdefault(doc_of(key), []).append(t)

    chunk_triples, doc_chunks = {}, {}
    for r in csv.DictReader(open(DATA / "contexts.csv", encoding="utf-8")):
        pid, doc = r["context_id"], doc_of(r["context_id"])
        chunk_triples[pid] = list(dict.fromkeys(by_node.get(pid, []) + by_doc.get(doc, [])))
        item = {"id": pid, "title": r["title"], "text": r["text"]}
        doc_chunks.setdefault(doc, []).append(item)
        # Mutual index by document number: the number in a title or heading, or the graph's own node id.
        keys = set(DOC_NO.findall(r["title"] + " " + r["text"][:300]))
        if not doc.startswith("doc-"):
            keys.add(doc)
        for key in keys:
            doc_chunks.setdefault(key, []).append(item)

    path = OUT / "triple_index.json"
    json.dump({"chunk_triples": chunk_triples, "doc_chunks": doc_chunks}, open(path, "w", encoding="utf-8"), ensure_ascii=False)
    with_t = sum(bool(v) for v in chunk_triples.values())
    print(f"triples {total}  passages {len(chunk_triples)}  passages with triples {with_t}  index keys {len(doc_chunks)}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vikag-faith/src/s4_judges.py
"""Step 4: LLM-based automatic metrics on the annotation sample.

ragas  - RAGAS faithfulness re-implemented from the library's two prompts
         (statement generation, then NLI verdicts); score = supported / statements
judge  - claim-level LLM judge over OUR claims, 3 labels (same label set as the humans)
Run with qwen2.5:3b (7b optional: pass it as an argument). Every metric sees the same evidence pool.
"""

import os
import sys

from common import BIG_MODEL, GEN_MODEL, OUT, append_jsonl, chat_json, read_jsonl

MAX_PASSAGE = 2500  # characters per passage inside LLM prompts

RAGAS_STATEMENTS = (
    "Given a question, an answer, and sentences from the answer analyze the complexity of each "
    "sentence given under 'sentences' and break down each sentence into one or more fully "
    "understandable statements while also ensuring no pronouns are used in each statement. "
    'Format the outputs in JSON: {"statements": ["...", "..."]}. Keep the statements in the '
    "language of the answer."
)
RAGAS_NLI = (
    "Your task is to judge the faithfulness of a series of statements based on a given context. "
    "For each statement you must return verdict as 1 if the statement can be directly inferred "
    "based on the context or 0 if the statement can not be directly inferred based on the context. "
    'Format the outputs in JSON: {"verdicts": [{"statement": "...", "reason": "...", "verdict": 1}]}'
)
JUDGE = (
    "Bạn là người kiểm chứng. Với mỗi mệnh đề (claim), so với BẰNG CHỨNG và gán đúng một nhãn:\n"
    "- supported: bằng chứng suy ra được mệnh đề.\n"
    "- contradicted: bằng chứng nói ngược lại mệnh đề (sai số liệu, ngày, tên văn bản, phủ định...).\n"
    "- unsupported: bằng chứng không đề cập, không đủ để kết luận.\n"
    'Trả về JSON: {"labels": [{"i": 1, "label": "supported"}, ...]} theo đúng thứ tự mệnh đề.'
)
LABELS = {"supported", "contradicted", "unsupported"}
JUDGE_SPAN = JUDGE.replace(
    'Trả về JSON: {"labels": [{"i": 1, "label": "supported"}, ...]} theo đúng thứ tự mệnh đề.',
    'Với mỗi mệnh đề, chép nguyên văn một câu ngắn trong BẰNG CHỨNG làm căn cứ vào trường span; nếu không có thì để chuỗi rỗng.\n'
    'Trả về JSON: {"labels": [{"i": 1, "label": "supported", "span": "..."}, ...]} theo đúng thứ tự mệnh đề.')
OUT_FILE = os.environ.get("VIKAG_JUDGE_FILE", "judges.jsonl")


def evidence_text(pool):
    parts = []
    for i, p in enumerate(pool, 1):
        facts = "; ".join(p["facts"])
        parts.append(f"[{i}] Văn bản: {p['title']}\n{p['text'][:MAX_PASSAGE]}\n(Thông tin đồ thị: {facts})")
    return "\n\n".join(parts)


def ragas(answer, question, context, model):
    sentences = [s.strip() for s in answer.replace("\n", " ").split(". ") if s.strip()]
    st = chat_json(
        f"question: {question}\nanswer: {answer}\nsentences: {sentences}",
        RAGAS_STATEMENTS, model, num_predict=700, num_ctx=4096,
    ).get("statements", [])
    st = [s for s in st if isinstance(s, str) and s.strip()]
    if not st:
        return {"statements": [], "verdicts": [], "score": None}
    listing = "\n".join(f"statement_{i}: {s}" for i, s in enumerate(st, 1))
    out = chat_json(f"context:\n{context}\n\nstatements:\n{listing}", RAGAS_NLI, model,
                    num_predict=1200, num_ctx=12288).get("verdicts", [])
    if not out:  # the call failed or returned nothing parseable: no score, not a zero
        return {"statements": st, "verdicts": [], "score": None}
    verdicts = [1 if str(v.get("verdict")).strip().lower() in {"1", "true", "yes"} else 0
                for v in out if isinstance(v, dict)][: len(st)]
    verdicts += [0] * (len(st) - len(verdicts))  # unparsed verdicts count as not inferred
    return {"statements": st, "verdicts": verdicts, "score": sum(verdicts) / len(st)}


def judge(claims, context, model):
    listing = "\n".join(f"{i}. {c}" for i, c in enumerate(claims, 1))
    out = chat_json(f"BẰNG CHỨNG:\n{context}\n\nMỆNH ĐỀ:\n{listing}", JUDGE, model,
                    num_predict=60 * len(claims) + 50, num_ctx=12288).get("labels", [])
    by_i = {}
    for x in out:
        if isinstance(x, dict) and str(x.get("label")).strip().lower() in LABELS:
            try:
                by_i[int(x.get("i"))] = str(x["label"]).strip().lower()
            except (TypeError, ValueError):
                pass
    # A claim the judge skipped gets None and is excluded from scoring, never a silent default.
    return [by_i.get(i) for i in range(1, len(claims) + 1)]


def judge_span(claims, context, model):
    """Small Qwen verifier that returns a label and an evidence span (not fine-tuned)."""
    listing = "\n".join(f"{i}. {c}" for i, c in enumerate(claims, 1))
    out = chat_json(f"BẰNG CHỨNG:\n{context}\n\nMỆNH ĐỀ:\n{listing}", JUDGE_SPAN, model,
                    num_predict=140 * len(claims) + 50, num_ctx=12288).get("labels", [])
    by_i = {}
    for x in out:
        if isinstance(x, dict) and str(x.get("label")).strip().lower() in LABELS:
            try:
                by_i[int(x.get("i"))] = {"label": str(x["label"]).strip().lower(), "span": str(x.get("span") or "")}
            except (TypeError, ValueError):
                pass
    return [by_i.get(i) for i in range(1, len(claims) + 1)]


def main_span(model):
    sample = {r["qid"] for r in read_jsonl(OUT / "sample.jsonl")}
    pools = {(p["qid"], p["system"]): p["pool"] for p in read_jsonl(OUT / "pools.jsonl")}
    claims = {}
    for c in read_jsonl(OUT / "claims.jsonl"):
        claims.setdefault((c["qid"], c["system"]), []).append(c)
    path = OUT / "judges_span.jsonl"
    done = {(r["qid"], r["system"]) for r in read_jsonl(path)} if path.exists() else set()
    keys = sorted(k for k in claims if k[0] in sample and k not in done)
    for n, key in enumerate(keys, 1):
        cl = claims[key]
        out = judge_span([c["claim"] for c in cl], evidence_text(pools[key]), model)
        append_jsonl(path, {"qid": key[0], "system": key[1], "model": model,
                            "judge": {c["cid"]: o for c, o in zip(cl, out)}})
        print(f"span {n}/{len(keys)}", flush=True)


def main(models):
    sample = {r["qid"] for r in read_jsonl(OUT / "sample.jsonl")}
    answers = {(a["qid"], a["system"]): a for a in read_jsonl(OUT / "answers.jsonl")}
    pools = {(p["qid"], p["system"]): p["pool"] for p in read_jsonl(OUT / "pools.jsonl")}
    claims = {}
    for c in read_jsonl(OUT / "claims.jsonl"):
        claims.setdefault((c["qid"], c["system"]), []).append(c)
    path = OUT / OUT_FILE
    done = {(r["qid"], r["system"], r["model"]) for r in read_jsonl(path)} if path.exists() else set()
    keys = sorted(k for k in answers if k[0] in sample)
    for model in models:
        for n, key in enumerate(keys, 1):
            if (*key, model) in done:
                continue
            a, cl = answers[key], claims[key]
            context = evidence_text(pools[key])
            r = ragas(a["answer"], a["question"], context, model)
            labels = judge([c["claim"] for c in cl], context, model)
            append_jsonl(path, {"qid": key[0], "system": key[1], "model": model, "ragas": r,
                                "judge": {c["cid"]: l for c, l in zip(cl, labels)}})
            print(f"{model} {n}/{len(keys)}", flush=True)


if __name__ == "__main__":
    if os.environ.get("VIKAG_SPAN") == "1":
        main_span(sys.argv[1])
    else:
        main(sys.argv[1:] or [GEN_MODEL])

In [ ]:
%%writefile /content/vikag-faith/src/s5_verifier.py
"""Step 5: claim verifier.

  python s5_verifier.py data     # synthetic regulatory pairs + ViNLI -> data/verifier_*.jsonl
  python s5_verifier.py train    # fine-tune mDeBERTa-v3-base-xnli -> models/verifier
  python s5_verifier.py predict  # entail/contra probabilities per claim, zero-shot and fine-tuned,
                                 # with evidence V1 (passage text), V2 (+ graph facts), V3 (+ claim-linked triples)
Thresholds that turn probabilities into labels are tuned later on the dev split (s7).
"""

import csv
import random
import re
import sys

import os

from common import DATA, OUT, ROOT, SEED, USE_CPU, read_jsonl, write_jsonl
from rules import overlap

BASE = os.environ.get("VIKAG_BASE", "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7")
# Second round: extra pairs and variants go to separate files, so the first round's results never change.
DATA_TAG = os.environ.get("VIKAG_DATA_TAG", "")
ZS_NAME = os.environ.get("VIKAG_ZS_NAME", "zs")
SCORE_FILE = os.environ.get("VIKAG_SCORE_FILE", "verifier_scores.jsonl")
SYNTH_TARGET = int(os.environ.get("VIKAG_SYNTH_TARGET", "0"))  # 0 = first-round sampling
# VIKAG_FT_TAG names a fine-tuned variant ("ft" = main run); VIKAG_NO_SYNTH=1 trains on ViNLI only (ablation).
FT_TAG = os.environ.get("VIKAG_FT_TAG", "ft")
FT_DIR = ROOT / "models" / ("verifier" if FT_TAG == "ft" else f"verifier_{FT_TAG}")
MAX_LEN = 320
WINDOW, STRIDE = 900, 700  # characters per evidence window
# Output order differs between checkpoints (mDeBERTa: E/N/C, XLM-R: C/N/E), so it is read from the config.
NLI_NAMES = {"supported": "entailment", "unsupported": "neutral", "contradicted": "contradiction"}


def head_ids(model):
    ids = {k.lower(): v for k, v in model.config.label2id.items()}
    if not all(v in ids for v in NLI_NAMES.values()):
        ids = {"entailment": 0, "neutral": 1, "contradiction": 2}
    return {k: ids[v] for k, v in NLI_NAMES.items()}


# ---------- data ----------

NEGATIONS = [("phải", "không phải"), ("được", "không được"), ("có", "không có"),
             ("bắt buộc", "không bắt buộc")]


def split_sentences(text):
    parts = re.split(r"(?<=[.;:])\s+|\n+", text)
    return [p.strip() for p in parts if 40 <= len(p.strip()) <= 300]


def corrupt(sentence, rng):
    """Make a contradiction by changing a number/date or inserting a negation."""
    nums = list(re.finditer(r"\d+", sentence))
    if nums and rng.random() < 0.7:
        m = rng.choice(nums)
        old = int(m.group())
        new = old + rng.choice([1, 2, 3, 5, 10]) * rng.choice([-1, 1])
        if new < 0 or new == old:
            new = old + 7
        return sentence[: m.start()] + str(new) + sentence[m.end():]
    for a, b in rng.sample(NEGATIONS, len(NEGATIONS)):
        hit = re.search(rf"\b{a}\b", sentence)
        if hit and not sentence[max(0, hit.start() - 6):hit.start()].strip().endswith("không"):
            return sentence[: hit.start()] + b + sentence[hit.end():]
    return None


def window_of(text, sentence):
    i = text.find(sentence)
    start = max(0, min(i - WINDOW // 3, len(text) - WINDOW))
    return text[start:start + WINDOW]


def build_data():
    from datasets import load_dataset

    rng = random.Random(SEED)
    sample = {r["qid"] for r in read_jsonl(OUT / "sample.jsonl")}
    held = {p["id"] for x in read_jsonl(OUT / "pools.jsonl") if x["qid"] in sample for p in x["pool"]}
    ctx = [r for r in csv.DictReader(open(DATA / "contexts.csv", encoding="utf-8"))
           if r["context_id"] not in held]
    by_doc = {}
    for r in ctx:
        by_doc.setdefault(r["title"], []).append(r)
    # First round: 2 sentences per passage, one pair of each kind. Second round: every sentence,
    # up to 2 corrupted and 2 unrelated hypotheses each, then cut to SYNTH_TARGET.
    per_passage, per_kind = (10 ** 6, 2) if SYNTH_TARGET else (2, 1)
    synth = []
    for r in ctx:
        sents = split_sentences(r["text"])
        rng.shuffle(sents)
        for s in sents[:per_passage]:
            premise = window_of(r["text"], s)
            if s not in premise:
                continue
            synth.append({"premise": premise, "hypothesis": s, "label": "supported", "src": "synth"})
            seen = {s}
            for _ in range(per_kind):
                bad = corrupt(s, rng)
                if bad and bad not in seen:
                    seen.add(bad)
                    synth.append({"premise": premise, "hypothesis": bad, "label": "contradicted", "src": "synth"})
            # Unsupported: a sentence from another passage (prefer the same document).
            for _ in range(per_kind):
                others = [o for o in by_doc[r["title"]] if o is not r] or rng.sample(ctx, 3)
                cand = split_sentences(rng.choice(others)["text"])
                if cand:
                    h = rng.choice(cand)
                    if h not in r["text"] and h not in seen:
                        seen.add(h)
                        synth.append({"premise": premise, "hypothesis": h, "label": "unsupported", "src": "synth"})
    names = {"entailment": "supported", "neutral": "unsupported", "contradiction": "contradicted"}
    vinli = load_dataset("presencesw/vinli_4_label")
    rows = {}
    for split in ("train", "dev"):
        rows[split] = [{"premise": x["sentence1"], "hypothesis": x["sentence2"],
                        "label": names[x["gold_label"]], "src": "vinli"}
                       for x in vinli[split] if x["gold_label"] in names]
    rng.shuffle(synth)
    if SYNTH_TARGET:
        synth = synth[:SYNTH_TARGET]
    n_dev = len(synth) // 10
    train = rng.sample(rows["train"], min(int(os.environ.get("VIKAG_VINLI", "4000")), len(rows["train"]))) + synth[n_dev:]
    dev = rng.sample(rows["dev"], 1000) + synth[:n_dev]
    rng.shuffle(train)
    train = train[: int(os.environ.get("VIKAG_MAX_TRAIN", len(train)))]  # smoke tests only
    write_jsonl(DATA / f"verifier_train{DATA_TAG}.jsonl", train)
    write_jsonl(DATA / f"verifier_dev{DATA_TAG}.jsonl", dev)
    count = lambda xs, k: {v: sum(x[k] == v for x in xs) for v in sorted({x[k] for x in xs})}  # noqa: E731
    print("train", len(train), count(train, "label"), count(train, "src"))
    print("dev", len(dev), count(dev, "label"), count(dev, "src"))


# ---------- train ----------

def train():
    import torch
    from torch.utils.data import DataLoader
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

    torch.manual_seed(SEED)
    dev_ = "cpu" if USE_CPU or not torch.cuda.is_available() else "cuda"
    tok = AutoTokenizer.from_pretrained(BASE)
    # The checkpoint is stored in float16; newer transformers keep that dtype, and pure fp16
    # training gave NaN loss on Colab from the first steps. Always train in float32.
    model = AutoModelForSequenceClassification.from_pretrained(BASE, torch_dtype=torch.float32).to(dev_)
    import transformers
    print("transformers", transformers.__version__, "torch", torch.__version__,
          "dtype", next(model.parameters()).dtype, flush=True)
    tr, dv = read_jsonl(DATA / f"verifier_train{DATA_TAG}.jsonl"), read_jsonl(DATA / f"verifier_dev{DATA_TAG}.jsonl")
    label_id = head_ids(model)
    if os.environ.get("VIKAG_NO_SYNTH") == "1":
        tr = [r for r in tr if r["src"] != "synth"]
    tr = tr[: int(os.environ.get("VIKAG_MAX_RUN", len(tr)))]  # XLM-R is slower on a free T4
    print("train rows", len(tr), "->", FT_DIR, flush=True)

    def collate(batch):
        enc = tok([b["premise"] for b in batch], [b["hypothesis"] for b in batch],
                  truncation="longest_first", max_length=MAX_LEN, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([label_id[b["label"]] for b in batch])
        return enc

    bs, accum, epochs = int(os.environ.get("VIKAG_BS", "4")), int(os.environ.get("VIKAG_ACCUM", "4")), 1
    loader = DataLoader(tr, batch_size=bs, shuffle=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=5e-6, weight_decay=0.01)
    steps = len(loader) * epochs // accum
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)

    def evaluate():
        model.eval()
        correct = {"vinli": [0, 0], "synth": [0, 0]}
        with torch.no_grad():
            for i in range(0, len(dv), 16):
                batch = dv[i:i + 16]
                enc = collate(batch).to(dev_)
                pred = model(**{k: v for k, v in enc.items() if k != "labels"}).logits.argmax(-1)
                for b, p, y in zip(batch, pred.tolist(), enc["labels"].tolist()):
                    correct[b["src"]][0] += p == y
                    correct[b["src"]][1] += 1
        model.train()
        return {k: round(c / max(n, 1), 4) for k, (c, n) in correct.items()}

    # A first Colab run collapsed to one class (dev 0.76 -> 0.34), so training keeps the best
    # dev checkpoint, starting from the base model itself: the result is never worse than zero-shot.
    FT_DIR.mkdir(parents=True, exist_ok=True)
    acc = evaluate()
    best = (acc["vinli"] + acc["synth"]) / 2
    print("dev acc before:", acc, flush=True)
    model.save_pretrained(FT_DIR); tok.save_pretrained(FT_DIR)
    model.train()
    nonfinite = 0
    for epoch in range(epochs):
        for i, enc in enumerate(loader):
            enc = enc.to(dev_)
            loss = model(**enc).loss / accum
            if not torch.isfinite(loss):
                nonfinite += 1
                if nonfinite > 20:
                    raise RuntimeError("loss is not finite; training aborted")
                continue
            loss.backward()
            if (i + 1) % accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
                if ((i + 1) // accum) % int(os.environ.get("VIKAG_EVAL_EVERY", "150")) == 0:
                    acc = evaluate()
                    score = (acc["vinli"] + acc["synth"]) / 2
                    kept = score > best
                    if kept:
                        best = score
                        model.save_pretrained(FT_DIR); tok.save_pretrained(FT_DIR)
                    print(f"step {(i + 1) // accum}/{steps} loss {loss.item() * accum:.4f} dev {acc}"
                          f"{' -> saved' if kept else ''}", flush=True)
    print("best dev mean acc:", round(best, 4), "non-finite batches skipped:", nonfinite, flush=True)


# ---------- predict ----------

def windows(text):
    if len(text) <= WINDOW:
        return [text]
    return [text[i:i + WINDOW] for i in range(0, len(text) - WINDOW // 3, STRIDE)]


def premises(pool, with_graph):
    out = []
    for p in pool:
        head = ""
        if with_graph:
            head = f"Văn bản: {p['title']}. " + " ".join(f + "." for f in p["facts"][:6]) + "\n"
        out += [head + w for w in windows(p["text"])]
    return out


MIN_LINK = 0.3  # share of the claim's words a triple must contain to count as linked
DOC_NO = re.compile(r"\d+/(?:\d{4}/)?[A-Za-zĐđ]+(?:-[A-Za-zĐđ]+)*")


def premises_v3(pool, claim, index):
    """V3: only the triples that share an entity with this claim, not every fact of every passage,
    plus source passages reached through the mutual index when the claim names another document."""
    cited = {m for m in DOC_NO.findall(claim) if m in index["doc_chunks"]}
    pool_ids = {p["id"] for p in pool}
    extra = []
    for doc in cited:
        extra += [c for c in index["doc_chunks"][doc] if c["id"] not in pool_ids]
    extra = sorted(extra, key=lambda c: -overlap(claim, c["text"]))[:2]
    out = []
    for p in pool + extra:
        triples = index["chunk_triples"].get(p["id"], [])
        ranked = sorted(triples, key=lambda t: -overlap(claim, t))
        picked = [t for t in ranked if overlap(claim, t) >= MIN_LINK][:3]
        head = ("Tri thức: " + " ".join(picked) + "\n") if picked else ""
        out += [head + w for w in windows(p["text"])]
    return out


def predict():
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    dev_ = "cpu" if USE_CPU or not torch.cuda.is_available() else "cuda"
    sample = {r["qid"] for r in read_jsonl(OUT / "sample.jsonl")}
    pools = {(p["qid"], p["system"]): p["pool"] for p in read_jsonl(OUT / "pools.jsonl")}
    claims = [c for c in read_jsonl(OUT / "claims.jsonl") if c["qid"] in sample]
    result = {c["cid"]: {} for c in claims}
    path_out = OUT / SCORE_FILE
    variants = os.environ.get("VIKAG_VARIANTS", "v1,v2").split(",")
    index = None
    if "v3" in variants:
        import json
        index = json.load(open(OUT / "triple_index.json", encoding="utf-8"))
    old = {r["cid"]: r for r in read_jsonl(path_out)} if path_out.exists() else {}
    # Variants already scored are skipped; VIKAG_FORCE=1 rescores both (zero-shot and FT_TAG).
    force = os.environ.get("VIKAG_FORCE") == "1"
    wanted = [(ZS_NAME, BASE), (FT_TAG, str(FT_DIR))]
    if os.environ.get("VIKAG_SKIP_ZS") == "1":
        wanted = wanted[1:]
    if os.environ.get("VIKAG_SKIP_FT") == "1":
        wanted = wanted[:1]
    runs = [(n, p) for n, p in wanted
            if force or not all(f"{n}_{v}" in old.get(c["cid"], {}) for c in claims for v in variants)]
    for name, path in runs:
        if name != ZS_NAME and not (FT_DIR / "config.json").exists():
            print("no fine-tuned model; zero-shot scores only")
            continue
        tok = AutoTokenizer.from_pretrained(path)
        model = AutoModelForSequenceClassification.from_pretrained(path, torch_dtype=torch.float32).to(dev_).eval()
        ids = head_ids(model)
        for variant in variants:
            for n, c in enumerate(claims):
                pool = pools[(c["qid"], c["system"])]
                prem = premises_v3(pool, c["claim"], index) if variant == "v3" else premises(pool, variant == "v2")
                probs = []
                with torch.no_grad():
                    size = int(os.environ.get("VIKAG_PRED_BS", "8"))
                    for i in range(0, len(prem), size):
                        enc = tok(prem[i:i + size], [c["claim"]] * len(prem[i:i + size]), truncation="longest_first",
                                  max_length=MAX_LEN, padding=True, return_tensors="pt").to(dev_)
                        probs += model(**enc).logits.softmax(-1).tolist()
                # Entailment: strongest window. Contradiction: only the 2 windows that share the
                # most words with the claim, since NLI calls unrelated text "contradiction" too.
                relevant = sorted(range(len(prem)), key=lambda i: -overlap(c["claim"], prem[i]))[:2]
                result[c["cid"]][f"{name}_{variant}"] = {
                    "entail": max(p[ids["supported"]] for p in probs),
                    "contra": max(probs[i][ids["contradicted"]] for i in relevant),
                    "contra_any": max(p[ids["contradicted"]] for p in probs)}
            print(name, variant, "done", flush=True)
        del model
        torch.cuda.empty_cache()
    merged = [{**{kk: vv for kk, vv in old.get(k, {}).items() if kk != "cid"}, **v} for k, v in result.items()]
    write_jsonl(path_out, [{"cid": k, **m} for k, m in zip(result, merged)])


if __name__ == "__main__":
    {"data": build_data, "train": train, "predict": predict}[sys.argv[1]]()

In [ ]:
%%writefile /content/vikag-faith/src/s6_annotation.py
"""Step 6: build the offline annotation page (outputs/annotation/annotate.html).

Annotators are blind to the system. Claude's pre-labels (outputs/prelabels.json) are pre-selected
and marked as suggestions; every change is saved in the browser and exported as JSON.
  python s6_annotation.py            # annotator A (corrects Claude's pre-labels)
  python s6_annotation.py --blank    # annotator B for the double-annotated subset, no pre-labels
  python s6_annotation.py --audit    # blind audit: 60 claims stratified by Claude label, no suggestions
  python s6_annotation.py --rest     # blind pass E: every sampled claim not in the audit, no suggestions
                                     # (pass D, same claims, was excluded: its export was not made through the page)
  python s6_annotation.py --rest100  # blind pass F: whole answers drawn at random until ~100 claims, no suggestions
"""

import hashlib
import json
import random
import sys

from common import OUT, SEED, read_jsonl

PAGE = r"""<!doctype html>
<html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Gán nhãn ViFaithKAG</title>
<style>
:root{--bg:#fafaf8;--card:#fff;--ink:#1d1d1b;--muted:#6b6b66;--line:#e2e1dc;--acc:#2b6cb0;--ok:#2f855a;--bad:#c53030;--warn:#b7791f}
body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.55 system-ui,"Segoe UI",sans-serif}
header{position:sticky;top:0;background:var(--card);border-bottom:1px solid var(--line);padding:10px 16px;display:flex;gap:12px;align-items:center;flex-wrap:wrap;z-index:2}
header b{font-size:16px}.prog{color:var(--muted)}button{font:inherit;padding:6px 12px;border:1px solid var(--line);border-radius:6px;background:var(--card);cursor:pointer}
button.primary{background:var(--acc);color:#fff;border-color:var(--acc)}
main{display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1fr);gap:16px;padding:16px;max-width:1400px;margin:auto}
@media(max-width:900px){main{grid-template-columns:1fr}}
.card{background:var(--card);border:1px solid var(--line);border-radius:8px;padding:14px 16px;margin-bottom:12px}
h3{margin:0 0 6px;font-size:13px;text-transform:uppercase;letter-spacing:.04em;color:var(--muted)}
.claim{border:1px solid var(--line);border-radius:8px;padding:10px 12px;margin:10px 0}
.claim.cur{border-color:var(--acc);box-shadow:0 0 0 2px #2b6cb033}
.claim .txt{font-weight:600;margin-bottom:6px}.opts{display:flex;gap:6px;flex-wrap:wrap}
.opts label{border:1px solid var(--line);border-radius:6px;padding:3px 10px;cursor:pointer}
.opts input{margin-right:4px}.sup{color:var(--ok)}.con{color:var(--bad)}.uns{color:var(--warn)}
.sugg{font-size:12px;color:var(--muted);margin-top:4px}.ev{margin-top:6px;font-size:13px;color:var(--muted)}
.ev input{width:5em}textarea{width:100%;font:inherit;font-size:13px;margin-top:6px;border:1px solid var(--line);border-radius:6px}
.pass{border-top:1px solid var(--line);padding:10px 0}.pass:first-child{border-top:0}
.ptitle{font-weight:600}.facts{font-size:12px;color:var(--muted);margin:2px 0 6px}.ptext{white-space:pre-wrap;font-size:14px}
.tag{font-size:11px;border-radius:4px;padding:1px 6px;margin-left:6px;background:#eef2f7;color:var(--muted)}
.guide{font-size:13px;color:var(--muted)}mark{background:#fff3b0}
</style></head><body>
<header><b>ViFaithKAG</b><span class="prog" id="prog"></span>
<button id="prev">← Trước</button><button id="next" class="primary">Tiếp →</button>
<select id="jump"></select><button id="export">Xuất kết quả (.json)</button>
<label class="guide"><input type="checkbox" id="hl" checked> tô sáng từ trùng</label></header>
<main><section><div class="card"><h3>Câu hỏi</h3><div id="q"></div></div>
<div class="card"><h3>Câu trả lời của hệ thống</h3><div id="a"></div></div>
<div class="card"><h3>Claim cần gán nhãn</h3><div class="guide">
<b class="sup">supported</b>: bằng chứng suy ra được claim ·
<b class="con">contradicted</b>: bằng chứng nói ngược lại (sai số, ngày, tên văn bản, phủ định) ·
<b class="uns">unsupported</b>: bằng chứng không đề cập / không đủ. Chỉ so với BẰNG CHỨNG bên phải, không dùng hiểu biết riêng.
Claim tách lỗi (không đứng độc lập, gộp nhiều ý) → đánh dấu "claim lỗi".</div><div id="claims"></div></div></section>
<section><div class="card"><h3>Bằng chứng (đánh số để ghi vào ô "đoạn")</h3><div id="pool"></div></div></section></main>
<script>
const DATA = __DATA__;
const MODE = "__MODE__";
const KEY = "vifaith-" + MODE;
let st = {};
try { st = JSON.parse(localStorage.getItem(KEY) || "{}"); } catch (e) { st = {}; }
let idx = Math.min(+(st._idx || 0), DATA.length - 1), cur = 0;
const save = () => { st._idx = idx; try { localStorage.setItem(KEY, JSON.stringify(st)); } catch (e) {} };
const esc = s => s.replace(/[&<>]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;"}[c]));
function words(s){ return new Set((s.toLowerCase().match(/[\p{L}\p{N}]+/gu)||[]).filter(w=>w.length>2)); }
function highlight(text, claim){
  if(!document.getElementById("hl").checked||!claim) return esc(text);
  const w = words(claim);
  return esc(text).replace(/[\p{L}\p{N}]+/gu, t => w.has(t.toLowerCase()) ? "<mark>"+t+"</mark>" : t);
}
function rec(cid, c){ if(!st[cid]) st[cid] = {label: MODE==="A" ? (c.pre||"") : "", evidence: MODE==="A" ? (c.pre_ev||"") : "", bad: MODE==="A" && !!c.pre_bad, note:"", touched:false}; return st[cid]; }
function done(){ let n=0,t=0; DATA.forEach(d=>d.claims.forEach(c=>{t++; const r=st[c.cid]; if(r&&r.label&&r.touched) n++;})); return [n,t]; }
function render(){
  const d = DATA[idx]; const [n,t] = done();
  document.getElementById("prog").textContent = `Câu ${idx+1}/${DATA.length} · đã xác nhận ${n}/${t} claim`;
  document.getElementById("q").textContent = d.question;
  document.getElementById("a").textContent = d.answer;
  const claimText = d.claims[cur] ? d.claims[cur].claim : "";
  document.getElementById("pool").innerHTML = d.pool.map((p,i)=>`<div class="pass"><div class="ptitle">[${i+1}] ${esc(p.title)}</div>
    <div class="facts">${p.facts.map(esc).join(" · ")}</div><div class="ptext">${highlight(p.text, claimText)}</div></div>`).join("");
  document.getElementById("claims").innerHTML = d.claims.map((c,i)=>{ const r = rec(c.cid,c);
    const opt = (v,cls)=>`<label class="${cls}"><input type="radio" name="l${i}" value="${v}" ${r.label===v?"checked":""}>${v}</label>`;
    return `<div class="claim ${i===cur?"cur":""}" data-i="${i}"><div class="txt">${i+1}. ${esc(c.claim)}</div>
      <div class="opts">${opt("supported","sup")}${opt("contradicted","con")}${opt("unsupported","uns")}
      <label><input type="checkbox" data-bad ${r.bad?"checked":""}>claim lỗi</label></div>
      ${MODE==="A"&&c.pre?`<div class="sugg">Gợi ý của Claude: <b>${c.pre}</b>${c.pre_note?" — "+esc(c.pre_note):""} ${r.touched?"· <b>đã xác nhận</b>":"· chưa xác nhận"}</div>`:""}
      <div class="ev">Đoạn bằng chứng (vd: 1,3): <input data-ev value="${esc(r.evidence||"")}"></div>
      <textarea rows="1" placeholder="Ghi chú (tuỳ chọn)" data-note>${esc(r.note||"")}</textarea></div>`; }).join("");
  document.querySelectorAll(".claim").forEach(el=>{ const i=+el.dataset.i, c=DATA[idx].claims[i], r=rec(c.cid,c);
    el.addEventListener("click",()=>{ if(cur!==i){cur=i; render();} });
    el.querySelectorAll("input[type=radio]").forEach(x=>x.addEventListener("change",()=>{r.label=x.value;r.touched=true;r.t=Date.now();save();render();}));
    el.querySelector("[data-bad]").addEventListener("change",e=>{r.bad=e.target.checked;r.touched=true;save();});
    el.querySelector("[data-ev]").addEventListener("change",e=>{r.evidence=e.target.value;r.touched=true;save();});
    el.querySelector("[data-note]").addEventListener("change",e=>{r.note=e.target.value;save();});
  });
  document.getElementById("jump").value = idx; save();
}
function go(k){ if(MODE!=="A" && k>0 && DATA[idx].claims.some(c=>!rec(c.cid,c).label) && !confirm("Trang này còn claim chưa chọn nhãn. Vẫn chuyển trang?")) return;
  if(MODE==="A") DATA[idx].claims.forEach(c=>{const r=rec(c.cid,c); if(r.label) r.touched=true;});
  idx=Math.max(0,Math.min(DATA.length-1,idx+k)); cur=0; render(); window.scrollTo(0,0); }
document.getElementById("next").onclick=()=>go(1); document.getElementById("prev").onclick=()=>go(-1);
document.getElementById("hl").onchange=render;
const j=document.getElementById("jump"); DATA.forEach((d,i)=>{const o=document.createElement("option");o.value=i;o.textContent="Câu "+(i+1);j.appendChild(o);});
j.onchange=()=>{idx=+j.value;cur=0;render();};
document.getElementById("export").onclick=()=>{ const out={mode:MODE,exported:new Date().toISOString(),labels:{}};
  DATA.forEach(d=>d.claims.forEach(c=>{const r=st[c.cid]; if(r) out.labels[c.cid]={label:r.label,evidence:r.evidence,bad:r.bad,note:r.note,confirmed:r.touched,t:r.t||null};}));
  const a=document.createElement("a"); a.href=URL.createObjectURL(new Blob([JSON.stringify(out,null,1)],{type:"application/json"}));
  a.download=`vifaith_labels_${MODE}.json`; a.click(); };
document.addEventListener("keydown",e=>{ if(e.target.tagName==="TEXTAREA"||e.target.tagName==="INPUT"&&e.target.type==="text") return;
  const c=DATA[idx].claims[cur]; if(!c) return; const r=rec(c.cid,c);
  const k={"1":"supported","2":"contradicted","3":"unsupported"}[e.key];
  if(k){r.label=k;r.touched=true;r.t=Date.now();save();render();}
  if(e.key==="j"){cur=Math.min(cur+1,DATA[idx].claims.length-1);render();}
  if(e.key==="k"){cur=Math.max(cur-1,0);render();}
  if(e.key==="ArrowRight")go(1); if(e.key==="ArrowLeft")go(-1); });
render();
</script></body></html>"""


def opaque(cid):
    """Claim IDs carry the system name; the page only sees an opaque ID (mapping kept aside)."""
    return hashlib.sha1(cid.encode()).hexdigest()[:12]


def audit_sample(pre, per_label={"supported": 30, "contradicted": 15, "unsupported": 15}):
    """Blind audit set: claims drawn per Claude label so agreement can be measured for every class."""
    rng = random.Random(SEED + 1)
    chosen = set()
    for label, n in per_label.items():
        pool = sorted(k for k, v in pre.items() if v["label"] == label and not v["bad"])
        chosen |= set(rng.sample(pool, min(n, len(pool))))
    return chosen


def main():
    blank = "--blank" in sys.argv
    audit = "--audit" in sys.argv
    rest = "--rest" in sys.argv
    rest100 = "--rest100" in sys.argv
    sample = read_jsonl(OUT / "sample.jsonl")
    keep = {r["qid"] for r in sample}
    if blank:  # annotator B: the dev questions only (double-annotated subset for kappa)
        keep = {r["qid"] for r in sample if r["split"] == "dev"}
    answers = {(a["qid"], a["system"]): a for a in read_jsonl(OUT / "answers.jsonl")}
    pools = {(p["qid"], p["system"]): p["pool"] for p in read_jsonl(OUT / "pools.jsonl")}
    pre = json.load(open(OUT / "prelabels.json", encoding="utf-8"))
    only = audit_sample(pre) if audit else None
    if rest:  # same blind protocol as the audit, on all claims the audit did not cover
        only = {c["cid"] for c in read_jsonl(OUT / "claims.jsonl") if c["qid"] in keep} - audit_sample(pre)
    if rest100:  # whole answers in random order until ~100 claims, so answer-level scores can use them too
        audited, by_answer = audit_sample(pre), {}
        for c in read_jsonl(OUT / "claims.jsonl"):
            if c["qid"] in keep and c["cid"] not in audited:
                by_answer.setdefault((c["qid"], c["system"]), []).append(c["cid"])
        keys = sorted(by_answer)
        random.Random(SEED + 2).shuffle(keys)
        only = set()
        for k in keys:
            if len(only) >= 100:
                break
            only |= set(by_answer[k])
    if blank or audit or rest or rest100:  # no suggestions: the annotator labels from scratch
        pre = {}
    claims = {}
    for c in read_jsonl(OUT / "claims.jsonl"):
        claims.setdefault((c["qid"], c["system"]), []).append(c)
    items = []
    for key in sorted(k for k in answers if k[0] in keep):
        a = answers[key]
        shown = [c for c in claims.get(key, []) if only is None or c["cid"] in only]
        if not shown:
            continue
        items.append({
            "question": a["question"], "answer": a["answer"],
            "pool": [{"title": p["title"], "facts": p["facts"], "text": p["text"]} for p in pools[key]],
            "claims": [{"cid": opaque(c["cid"]), "claim": c["claim"],
                        "pre": pre.get(c["cid"], {}).get("label", ""),
                        "pre_ev": pre.get(c["cid"], {}).get("evidence", ""),
                        "pre_note": pre.get(c["cid"], {}).get("note", ""),
                        "pre_bad": pre.get(c["cid"], {}).get("bad", False)} for c in shown],
        })
    random.Random(SEED).shuffle(items)  # interleave systems; annotators never see the system name
    mode = "F" if rest100 else "E" if rest else "C" if audit else "B" if blank else "A"
    html = PAGE.replace("__DATA__", json.dumps(items, ensure_ascii=False)).replace("__MODE__", mode)
    out = OUT / "annotation" / f"annotate_{mode}.html"
    out.parent.mkdir(exist_ok=True)
    out.write_text(html, encoding="utf-8")
    ids = {opaque(c["cid"]): c["cid"] for cl in claims.values() for c in cl}
    (out.parent / "id_map.json").write_text(json.dumps(ids, ensure_ascii=False, indent=1), encoding="utf-8")
    print(out, len(items), "answers", sum(len(i["claims"]) for i in items), "claims")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vikag-faith/src/s6_quick.py
"""Quick blind check (pass G): 30 random claims, one per screen, with only the evidence sentences that share
the most words with the claim (the full passages stay one click away). No suggestions, no system names.

  python s6_quick.py        ->  outputs/annotation/annotate_G.html  (export: vifaith_labels_G.json)
  python s6_quick.py H      ->  pass H: 30 further random claims, none already labelled in passes C, G

Made because the full page (pass F) was too hard to use; s7 reports these labels as an extra audit sample,
separately from pass C, since the annotator saw short excerpts instead of whole passages.
"""

import json
import random
import re
import sys

from common import OUT, SEED, read_jsonl
from s6_annotation import audit_sample, opaque

N_CLAIMS, N_SENTENCES = 30, 4

PAGE = r"""<!doctype html>
<html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Kiểm tra nhanh 30 ý (lượt __PASS__)</title>
<style>
:root{--bg:#f6f8f8;--card:#fff;--ink:#16232a;--muted:#56656c;--line:#d7e0e1;--acc:#0e5e6f;--ok:#22744f;--bad:#b4432f;--warn:#9a6a08;--hl:#fff0a8}
body{margin:0;background:var(--bg);color:var(--ink);font:17px/1.6 system-ui,"Segoe UI",sans-serif}
.wrap{max-width:760px;margin:0 auto;padding:20px 16px 40px;display:flex;flex-direction:column;gap:14px}
.bar{height:8px;background:var(--line);border-radius:4px;overflow:hidden}.bar i{display:block;height:100%;background:var(--acc)}
.top{display:flex;justify-content:space-between;align-items:center;font-size:14px;color:var(--muted)}
.card{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:18px 20px}
.lab{font-size:13px;font-weight:600;letter-spacing:.05em;text-transform:uppercase;color:var(--muted);margin-bottom:6px}
.claim{font-size:21px;font-weight:600;line-height:1.45}
.q{font-size:14px;color:var(--muted)}
.sent{border-left:3px solid var(--line);padding:6px 12px;margin:8px 0;font-size:16px}
.sent .t{display:block;font-size:12.5px;color:var(--muted)}
mark{background:var(--hl);border-radius:2px;padding:0 1px}
details summary{cursor:pointer;color:var(--acc);font-size:14.5px;margin-top:6px}
.full{white-space:pre-wrap;font-size:14px;max-height:300px;overflow:auto;background:var(--bg);border-radius:6px;padding:8px 10px;margin-top:6px}
.btns{display:grid;gap:10px}
.btns button{font:600 17px system-ui,sans-serif;border-radius:10px;padding:14px 16px;cursor:pointer;text-align:left;border:2px solid var(--line);background:var(--card);color:var(--ink)}
.btns button small{display:block;font-weight:400;font-size:13.5px;color:var(--muted)}
.btns button.sel{border-color:var(--acc);background:#dcebee}
.b1{border-left:8px solid var(--ok)!important}.b2{border-left:8px solid var(--bad)!important}.b3{border-left:8px solid var(--warn)!important}
.row{display:flex;justify-content:space-between;gap:10px;flex-wrap:wrap;align-items:center}
.link{background:none;border:none;color:var(--muted);font-size:14px;cursor:pointer;text-decoration:underline;padding:0}
.nav button{font:600 15px system-ui;border:1px solid var(--line);background:var(--card);border-radius:8px;padding:8px 14px;cursor:pointer}
.done{text-align:center}.done button{font:700 18px system-ui;background:var(--acc);color:#fff;border:none;border-radius:10px;padding:14px 22px;cursor:pointer}
button:focus-visible{outline:3px solid var(--acc);outline-offset:2px}
</style></head><body><div class="wrap" id="app"></div>
<script>
const DATA = __DATA__;
const KEY = "vifaith-__PASS__";
let st = {};
try { st = JSON.parse(localStorage.getItem(KEY) || "{}"); } catch (e) {}
let i = Math.min(st._i || 0, DATA.length);
const save = () => { st._i = i; try { localStorage.setItem(KEY, JSON.stringify(st)); } catch (e) {} };
const esc = (s) => s.replace(/[&<>]/g, (c) => ({"&": "&amp;", "<": "&lt;", ">": "&gt;"}[c]));
const words = (s) => new Set((s.toLowerCase().match(/[\p{L}\p{N}]+/gu) || []).filter((w) => w.length > 2 || /\d/.test(w)));
const hl = (text, claim) => { const w = words(claim); return esc(text).replace(/[\p{L}\p{N}]+/gu, (t) => w.has(t.toLowerCase()) ? `<mark>${t}</mark>` : t); };
const OPTS = [["supported", "Văn bản nói đúng như vậy", "có câu cùng ý, kể cả khi khác chữ", "b1"],
              ["contradicted", "Văn bản nói khác", "số, ngày, tên cơ quan khác, hoặc nói ngược lại", "b2"],
              ["unsupported", "Văn bản không nhắc tới", "tìm không thấy trong các câu bên trên", "b3"]];
const nDone = () => DATA.filter((d) => st[d.cid] && (st[d.cid].label || st[d.cid].bad)).length;
function pick(label, bad) {
  const d = DATA[i]; const r = st[d.cid] || {};
  st[d.cid] = {...r, label: bad ? "" : label, bad: !!bad, confirmed: true, t: Date.now()};
  i++; save(); render(); window.scrollTo(0, 0);
}
function render() {
  const app = document.getElementById("app"), n = nDone();
  const head = `<div class="top"><span>Đã làm ${n}/${DATA.length}</span><span>${n > 0 && i < DATA.length ? `<button class="link" onclick="exportJson()">Lưu tạm (làm dở)</button> · ` : ""}${i > 0 ? `<button class="link" onclick="i--;save();render()">← Ý trước</button>` : ""}</span></div>
    <div class="bar"><i style="width:${100 * n / DATA.length}%"></i></div>`;
  if (i >= DATA.length) {
    app.innerHTML = head + `<div class="card done"><p style="font-size:20px;font-weight:600">Xong ${n}/${DATA.length} ý. Cảm ơn bạn!</p>
      <p>Bấm nút dưới để lưu kết quả (file <b>vifaith_labels___PASS__.json</b> vào thư mục Downloads).</p>
      <button id="exp">Lưu kết quả</button><p style="font-size:14px;color:var(--muted)">Muốn sửa ý nào thì bấm “← Ý trước”.</p></div>`;
    document.getElementById("exp").onclick = exportJson; return;
  }
  const d = DATA[i], r = st[d.cid] || {};
  const sents = d.sents.map((s) => `<div class="sent"><span class="t">${esc(s.title)}</span>${hl(s.text, d.claim)}</div>`).join("");
  const full = d.full.map((p) => `<p style="font-weight:600;font-size:13.5px;margin:8px 0 0">${esc(p.title)}</p><div class="full">${hl(p.text, d.claim)}</div>`).join("");
  app.innerHTML = head + `
   <div class="card"><div class="lab">Ý ${i + 1} cần kiểm tra</div><div class="claim">${esc(d.claim)}</div>
     <p class="q">Trích từ câu trả lời cho câu hỏi: ${esc(d.question)}</p></div>
   <div class="card"><div class="lab">Văn bản gốc (các câu liên quan nhất, từ trùng được tô vàng)</div>${sents}
     <details id="more"><summary>Không thấy? Xem toàn bộ văn bản</summary>${full}</details></div>
   <div class="btns">${OPTS.map(([v, t, s, c], k) => `<button class="${c}${r.label === v ? " sel" : ""}" onclick="pick('${v}')">${k + 1}. ${t}<small>${s}</small></button>`).join("")}</div>
   <div class="row"><button class="link" onclick="pick('',true)">Ý này khó hiểu, không đọc ra nghĩa (bỏ qua)</button></div>`;
  document.getElementById("more").ontoggle = (e) => { st[d.cid] = {...(st[d.cid] || {}), expanded: true}; save(); };
}
function exportJson() {
  const out = {mode: "__PASS__", exported: new Date().toISOString(), labels: {}};
  DATA.forEach((d) => { const r = st[d.cid]; if (r) out.labels[d.cid] = {label: r.label || "", bad: !!r.bad, confirmed: !!r.confirmed, t: r.t || null, expanded: !!r.expanded, evidence: "", note: ""}; });
  const a = document.createElement("a"); a.href = URL.createObjectURL(new Blob([JSON.stringify(out, null, 1)], {type: "application/json"}));
  a.download = "vifaith_labels___PASS__.json"; a.click();
}
document.addEventListener("keydown", (e) => { const k = {"1": "supported", "2": "contradicted", "3": "unsupported"}[e.key]; if (k && i < DATA.length) pick(k); });
render();
</script></body></html>"""


def words(s):
    return {w for w in re.findall(r"\w+", s.lower()) if len(w) > 2 or any(ch.isdigit() for ch in w)}


def best_sentences(claim, pool):
    cw = words(claim)
    cands = []
    for p in pool:
        for s in re.split(r"(?<=[.;:])\s+|\n+", p["text"]):
            s = " ".join(s.split())
            if 25 <= len(s):
                cands.append((len(cw & words(s)), -len(s), p["title"], s[:420]))
    cands.sort(reverse=True)
    return [{"title": t, "text": s} for _, _, t, s in cands[:N_SENTENCES]]


QUICK_PASSES = "GHIJ"  # each later pass skips every claim labelled in an earlier one


def labelled(passes):
    ids = json.load(open(OUT / "annotation" / "id_map.json", encoding="utf-8"))
    done = set()
    for name in passes:
        path = OUT / "annotation" / f"vifaith_labels_{name}.json"
        if path.exists():
            done |= {ids[k] for k in json.load(open(path, encoding="utf-8"))["labels"] if k in ids}
    return done


def main(name="G"):
    assert name in QUICK_PASSES, f"pass must be one of {QUICK_PASSES}"
    pre = json.load(open(OUT / "prelabels.json", encoding="utf-8"))
    sample = {r["qid"] for r in read_jsonl(OUT / "sample.jsonl")}
    answers = {(a["qid"], a["system"]): a for a in read_jsonl(OUT / "answers.jsonl")}
    pools = {(p["qid"], p["system"]): p["pool"] for p in read_jsonl(OUT / "pools.jsonl")}
    taken = audit_sample(pre) | labelled("C" + QUICK_PASSES[:QUICK_PASSES.index(name)])
    claims = [c for c in read_jsonl(OUT / "claims.jsonl") if c["qid"] in sample and c["cid"] not in taken]
    # Pass G keeps its original draw (seed + 3); later passes draw from what is left.
    chosen = random.Random(SEED + 3 + QUICK_PASSES.index(name)).sample(claims, min(N_CLAIMS, len(claims)))
    items = []
    for c in chosen:
        key = (c["qid"], c["system"])
        items.append({"cid": opaque(c["cid"]), "claim": c["claim"], "question": answers[key]["question"],
                      "sents": best_sentences(c["claim"], pools[key]),
                      "full": [{"title": p["title"], "text": p["text"]} for p in pools[key]]})
    out = OUT / "annotation" / f"annotate_{name}.html"
    out.write_text(PAGE.replace("__DATA__", json.dumps(items, ensure_ascii=False)).replace("__PASS__", name), encoding="utf-8")
    print(out, len(items), "claims")


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else "G")

In [ ]:
%%writefile /content/vikag-faith/src/s7_analysis.py
"""Step 7: meta-evaluation of faithfulness metrics against the gold claim labels.

Gold = HUMAN labels from the two blind passes (C: 60-claim audit, F: ~100 more claims) once
vifaith_labels_F.json exists; Claude's labels then become one more method ("claude_silver"). Without D,
gold = Claude's labels, reported as SILVER. VIKAG_GOLD=silver forces the silver setting.
Writes outputs/analysis/results.json.
"""

import json
import os
import random
from collections import Counter, defaultdict

import numpy as np
from scipy.stats import kendalltau, spearmanr, wilcoxon
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score, roc_auc_score
from vihermes.evaluation import token_f1
from vihermes.retrieval import tokens

from common import OUT, SEED, read_jsonl
from rules import check

LABELS = ["supported", "contradicted", "unsupported"]
ANN = OUT / "annotation"
RES = OUT / "analysis"
RES.mkdir(exist_ok=True)


def load_human(name):
    path = ANN / f"vifaith_labels_{name}.json"
    if not path.exists():
        return None
    ids = json.load(open(ANN / "id_map.json", encoding="utf-8"))
    raw = json.load(open(path, encoding="utf-8"))["labels"]
    # A claim marked malformed may carry no label; keep it so it is excluded rather than missing.
    return {ids[k]: v for k, v in raw.items() if (v.get("label") or v.get("bad")) and ids.get(k)}


def gold_labels():
    """Human gold = blind passes C + F (never pass A: suggestions confirmed without a single change in
    ~10 minutes). Falls back to Claude's silver labels while pass F is missing."""
    pre = json.load(open(OUT / "prelabels.json", encoding="utf-8"))
    # Pass D is excluded (its export was not produced through the annotation page) and so is pass E,
    # which re-typed D. Pass F is the blind relabel of ~100 claims.
    d = load_human("F")
    if d and os.environ.get("VIKAG_GOLD", "human") == "human":
        human = {**(load_human("C") or {}), **d}
        return {k: {"label": v["label"], "bad": bool(v.get("bad"))} for k, v in human.items()}, "human", pre, None
    return {k: {"label": v["label"], "bad": v["bad"]} for k, v in pre.items()}, "silver", pre, None


def decide(entail, contra, ts, tc):
    if entail >= ts:
        return "supported"
    if contra >= tc:
        return "contradicted"
    return "unsupported"


def apply_rule(label, claim, evidence):
    r = check(claim, evidence)
    if r == "contradicted":
        return "contradicted"
    if r == "unsupported" and label == "supported":
        return "unsupported"
    return label


def macro_f1(y, p):
    return f1_score(y, p, labels=LABELS, average="macro", zero_division=0)


def bootstrap(y, p, n=1000):
    rng = np.random.default_rng(SEED)
    y, p = np.array(y), np.array(p)
    vals = [macro_f1(y[i], p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def main():
    sample = {r["qid"]: r["split"] for r in read_jsonl(OUT / "sample.jsonl")}
    claims = {c["cid"]: c for c in read_jsonl(OUT / "claims.jsonl") if c["qid"] in sample}
    pools = {(p["qid"], p["system"]): p["pool"] for p in read_jsonl(OUT / "pools.jsonl")}
    answers = read_jsonl(OUT / "answers.jsonl")
    judges = read_jsonl(OUT / "judges.jsonl")
    scores = {r["cid"]: r for r in read_jsonl(OUT / "verifier_scores.jsonl")}
    # SelfCheckGPT (s9) scores sit beside the verifier's, under the variant name "selfcheck".
    if (OUT / "selfcheck_scores.jsonl").exists():
        for r in read_jsonl(OUT / "selfcheck_scores.jsonl"):
            if r["cid"] in scores:
                scores[r["cid"]]["selfcheck"] = {"entail": r["entail"], "contra": r["contra"]}
    gold, gold_kind, pre, reviewed = gold_labels()

    def evidence(c):
        return [f"{p['title']} {' '.join(p['facts'])} {p['text']}" for p in pools[(c["qid"], c["system"])]]

    unlabeled = [k for k in claims if k not in gold]
    usable = [k for k in claims if k in gold and not gold[k]["bad"]]
    split = {k: sample[claims[k]["qid"]] for k in usable}
    dev = [k for k in usable if split[k] == "dev"]
    test = [k for k in usable if split[k] == "test"]

    # ---- claim-level predictions per method ----
    preds = {}
    for model in ("qwen2.5:3b", "qwen2.5:7b"):
        tag = "judge_" + model.split(":")[1]
        preds[tag] = {}
        for j in judges:
            if j["model"] == model:
                preds[tag].update({k: v for k, v in j["judge"].items() if v})
    thresholds = {}
    grid = np.round(np.arange(0.1, 0.95, 0.05), 2)
    for variant in ("zs_v1", "zs_v2", "ft_v1", "ft_v2", "nosynth_v1", "nosynth_v2", "selfcheck"):
        if not all(variant in scores[k] for k in usable):
            if any(variant in scores[k] for k in usable):
                print("skipping", variant, "- scores missing for some claims")
            continue
        best = max(((ts, tc) for ts in grid for tc in grid),
                   key=lambda t: macro_f1([gold[k]["label"] for k in dev],
                                          [decide(scores[k][variant]["entail"], scores[k][variant]["contra"], *t) for k in dev]))
        thresholds[variant] = [float(x) for x in best]
        base = {k: decide(scores[k][variant]["entail"], scores[k][variant]["contra"], *best) for k in usable}
        preds["nli_" + variant] = base
        if variant != "selfcheck":  # SelfCheckGPT is reference-free; the rule needs the evidence
            preds["nli_" + variant + "+rule"] = {k: apply_rule(v, claims[k]["claim"], evidence(claims[k])) for k, v in base.items()}
    if gold_kind == "human":  # Claude's labels are now one more automatic method
        preds["claude_silver"] = {k: pre[k]["label"] for k in usable if not pre[k]["bad"]}
    preds["rule_only"] = {k: (check(claims[k]["claim"], evidence(claims[k])) or "supported") for k in usable}
    preds["majority_supported"] = {k: "supported" for k in usable}

    same_ft = all(scores[k].get("ft_v1") == scores[k].get("zs_v1") for k in usable)
    same_nosynth = all(scores[k].get("nosynth_v1") == scores[k].get("zs_v1") for k in usable) \
        if "nli_nosynth_v1" in preds else None

    claim_table = {}
    for name, p in preds.items():
        ks = [k for k in test if k in p]
        y = [gold[k]["label"] for k in ks]
        yp = [p[k] for k in ks]
        row = {"n": len(ks), "coverage": len(ks) / len(test), "acc": accuracy_score(y, yp),
               "macro_f1": macro_f1(y, yp), "ci95": bootstrap(y, yp),
               "f1_per_class": dict(zip(LABELS, f1_score(y, yp, labels=LABELS, average=None, zero_division=0).tolist()))}
        # Detecting "not supported" claims: NLI uses 1 - entail as a score, others their hard label.
        yb = [int(v != "supported") for v in y]
        if name.startswith("nli_") and "+rule" not in name:
            sb = [1 - scores[k][name[4:]]["entail"] for k in ks]
        else:
            sb = [int(v != "supported") for v in yp]
        row["auroc_not_supported"] = roc_auc_score(yb, sb) if 0 < sum(yb) < len(yb) else None
        claim_table[name] = row

    # Paired bootstrap of macro-F1 differences on the claims both methods labelled.
    def paired(a, b, n=1000):
        ks = [k for k in test if k in preds[a] and k in preds[b]]
        y = np.array([gold[k]["label"] for k in ks])
        pa, pb = np.array([preds[a][k] for k in ks]), np.array([preds[b][k] for k in ks])
        rng = np.random.default_rng(SEED)
        d = [macro_f1(y[i], pa[i]) - macro_f1(y[i], pb[i]) for i in (rng.integers(0, len(ks), len(ks)) for _ in range(n))]
        return {"n": len(ks), "diff": macro_f1(y, pa) - macro_f1(y, pb),
                "ci95": [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))], "p_le_0": float(np.mean(np.array(d) <= 0))}

    comparisons = {f"{a} - {b}": paired(a, b) for a, b in (
        ("nli_ft_v1", "judge_7b"), ("nli_ft_v1", "nli_zs_v1"), ("nli_ft_v1", "nli_nosynth_v1"),
        ("nli_nosynth_v1", "nli_zs_v1"), ("nli_ft_v1", "nli_ft_v2"), ("judge_7b", "nli_selfcheck"))
        if a in preds and b in preds}

    # ---- answer-level: GGF / CR / UR ----
    by_answer = defaultdict(list)
    for k in usable:
        by_answer[(claims[k]["qid"], claims[k]["system"])].append(k)

    def rates(ks, lab):
        c = Counter(lab[k] for k in ks)
        n = len(ks)
        return c["supported"] / n, c["contradicted"] / n, c["unsupported"] / n

    human = {a: rates(ks, {k: gold[k]["label"] for k in ks}) for a, ks in by_answer.items()}
    test_answers = [a for a in by_answer if sample[a[0]] == "test"]
    corr = {}
    for name, p in preds.items():
        if name == "majority_supported":
            continue
        xs, ys = [], []
        for a in test_answers:
            ks = [k for k in by_answer[a] if k in p]
            if ks:
                xs.append(rates(ks, p)[0])
                ys.append(human[a][0])
        corr[name] = {"n": len(xs), "spearman": spearmanr(xs, ys).statistic, "kendall": kendalltau(xs, ys).statistic}
    for j_model in ("qwen2.5:3b", "qwen2.5:7b"):
        xs, ys = [], []
        for j in judges:
            a = (j["qid"], j["system"])
            if j["model"] == j_model and a in test_answers and j["ragas"]["score"] is not None:
                xs.append(j["ragas"]["score"])
                ys.append(human[a][0])
        corr["ragas_" + j_model.split(":")[1]] = {"n": len(xs), "spearman": spearmanr(xs, ys).statistic,
                                                   "kendall": kendalltau(xs, ys).statistic}

    # ---- system comparison (all 80 sampled answers, human/silver gold) ----
    systems = {}
    for s in ("naive", "graph"):
        rs = [human[a] for a in human if a[1] == s]
        systems[s] = {"answers": len(rs), "GGF": float(np.mean([r[0] for r in rs])),
                      "CR": float(np.mean([r[1] for r in rs])), "UR": float(np.mean([r[2] for r in rs]))}
    paired = [(human[(q, "naive")][0], human[(q, "graph")][0]) for q in sample
              if (q, "naive") in human and (q, "graph") in human]
    systems["wilcoxon_GGF_p"] = float(wilcoxon([x for x, _ in paired], [y for _, y in paired]).pvalue) \
        if any(x != y for x, y in paired) else None
    hop = defaultdict(lambda: defaultdict(list))
    for (q, s), r in human.items():
        hop[s][claims[by_answer[(q, s)][0]]["hop"]].append(r)
    systems["by_hop"] = {s: {h: {"n": len(v), "GGF": float(np.mean([x[0] for x in v])),
                                 "CR": float(np.mean([x[1] for x in v]))} for h, v in sorted(d.items())}
                         for s, d in hop.items()}

    # ---- RQ3: verify-then-revise (s8). Before/after are both scored by the 7B claim judge, the
    # same judge that flagged the claims, so the gain is an upper bound; token F1 is independent. ----
    revise = None
    if (OUT / "revise.jsonl").exists():
        ans = {(a["qid"], a["system"]): a for a in answers}
        j7 = {(j["qid"], j["system"]): j["judge"] for j in judges if j["model"] == "qwen2.5:7b"}

        def judged(labels):
            ls = [x for x in labels if x]
            if not ls:
                return None
            c = Counter(ls)
            return c["supported"] / len(ls), c["contradicted"] / len(ls), c["unsupported"] / len(ls)

        def em(pred, ref):
            return float(tokens(pred) == tokens(ref))

        def recall(pred, ref):  # share of the reference's tokens the answer keeps
            p, r = Counter(tokens(pred)), Counter(tokens(ref))
            return sum((p & r).values()) / max(sum(r.values()), 1)

        rev_rows = read_jsonl(OUT / "revise.jsonl")

        def build(max_rounds):
            """One row per answer, stopping the revise loop after `max_rounds` rounds (ablation on rounds)."""
            rows = []
            for r in rev_rows:
                a, rounds = (r["qid"], r["system"]), r["rounds"][:max_rounds]
                before = judged(list(j7.get(a, {}).values()))
                last = rounds[-1] if rounds else None
                after = judged(last["labels"]) if last else before
                text, ref = (last["answer"] if last else ans[a]["answer"]), ans[a]["gold_answer"]
                rows.append({"a": a, "revised": bool(rounds), "rounds": len(rounds), "before": before, "after": after,
                             "f1_before": token_f1(ans[a]["answer"], ref), "f1_after": token_f1(text, ref),
                             "em_before": em(ans[a]["answer"], ref), "em_after": em(text, ref),
                             "recall_before": recall(ans[a]["answer"], ref), "recall_after": recall(text, ref),
                             "abstain_after": not (last["claims"] if last else True),
                             "claims_before": len(j7.get(a, {})), "claims_after": len(last["claims"]) if last else len(j7.get(a, {}))})
            return rows

        rows = build(2)

        def summary(rs):
            ok = [x for x in rs if x["before"] and x["after"]]
            out = {"answers": len(rs), "revised": sum(x["revised"] for x in rs),
                   "mean_rounds_revised": float(np.mean([x["rounds"] for x in rs if x["revised"]])) if any(x["revised"] for x in rs) else 0.0,
                   "abstained_after": sum(x["abstain_after"] for x in rs), "scored": len(ok)}
            for i, m in enumerate(("GGF", "CR", "UR")):
                out[m + "_before"] = float(np.mean([x["before"][i] for x in ok]))
                out[m + "_after"] = float(np.mean([x["after"][i] for x in ok]))
            for m in ("f1", "em", "recall"):
                out[f"answer_{m}_before"] = float(np.mean([x[m + "_before"] for x in rs]))
                out[f"answer_{m}_after"] = float(np.mean([x[m + "_after"] for x in rs]))
            out["claims_before"] = float(np.mean([x["claims_before"] for x in rs]))
            out["claims_after"] = float(np.mean([x["claims_after"] for x in rs]))
            diff = [(x["before"][0], x["after"][0]) for x in ok if x["revised"]]
            out["wilcoxon_GGF_p"] = float(wilcoxon([b for b, _ in diff], [c for _, c in diff]).pvalue) \
                if any(b != c for b, c in diff) else None
            fd = [(x["f1_before"], x["f1_after"]) for x in rs if x["revised"]]
            out["wilcoxon_F1_p"] = float(wilcoxon([b for b, _ in fd], [c for _, c in fd]).pvalue) \
                if any(b != c for b, c in fd) else None
            return out

        revise = {"all": summary(rows), **{s: summary([x for x in rows if x["a"][1] == s]) for s in ("naive", "graph")}}
        # Ablation on the number of rounds: stop after round 1 vs. allow round 2, all answers, and the
        # answers that actually needed a second round (still flagged after round 1).
        one = build(1)
        two_ids = sorted({r2["a"] for r2 in rows if r2["rounds"] == 2})
        revise["by_max_rounds"] = {"1": summary(one), "2": revise["all"],
                                   "needed_round_2": {"1": summary([x for x in one if x["a"] in two_ids]),
                                                      "2": summary([x for x in rows if x["a"] in two_ids])}}
        r1, r2 = {x["a"]: x for x in one if x["a"] in two_ids}, {x["a"]: x for x in rows if x["a"] in two_ids}
        both = [a for a in two_ids if r1[a]["after"] and r2[a]["after"]]
        pair = lambda xs, ys: float(wilcoxon(xs, ys).pvalue) if any(p != q for p, q in zip(xs, ys)) else None  # noqa: E731
        revise["by_max_rounds"]["round2_vs_round1"] = {
            "n": len(both),
            "GGF_r1": float(np.mean([r1[a]["after"][0] for a in both])), "GGF_r2": float(np.mean([r2[a]["after"][0] for a in both])),
            "CR_r1": float(np.mean([r1[a]["after"][1] for a in both])), "CR_r2": float(np.mean([r2[a]["after"][1] for a in both])),
            "F1_r1": float(np.mean([r1[a]["f1_after"] for a in two_ids])), "F1_r2": float(np.mean([r2[a]["f1_after"] for a in two_ids])),
            "recall_r1": float(np.mean([r1[a]["recall_after"] for a in two_ids])), "recall_r2": float(np.mean([r2[a]["recall_after"] for a in two_ids])),
            "wilcoxon_GGF_p": pair([r1[a]["after"][0] for a in both], [r2[a]["after"][0] for a in both]),
            "wilcoxon_F1_p": pair([r1[a]["f1_after"] for a in two_ids], [r2[a]["f1_after"] for a in two_ids])}

    # ---- QA quality on all 100 questions (context for the faithfulness numbers) ----
    qa = {}
    for s in ("naive", "graph"):
        xs = [a for a in answers if a["system"] == s]
        qa[s] = {"questions": len(xs),
                 "answer_f1": float(np.mean([token_f1(a["answer"], a["gold_answer"]) for a in xs])),
                 "gold_passage_recall": float(np.mean([len({p["id"] for p in a["retrieved"]} & set(a["gold_ids"])) / len(a["gold_ids"]) for a in xs])),
                 "abstain_rate": float(np.mean(["không tìm thấy" in a["answer"].lower() or "không có thông tin" in a["answer"].lower() for a in xs]))}

    # ---- agreement ----
    agreement = {}
    if reviewed:
        ks = sorted(reviewed)
        agreement["claude_vs_human_kappa"] = cohen_kappa_score([pre[k]["label"] for k in ks], [gold[k]["label"] for k in ks])
        agreement["claude_vs_human_acc"] = float(np.mean([pre[k]["label"] == gold[k]["label"] for k in ks]))
        agreement["n"] = len(ks)
        agreement["changed"] = Counter(f"{pre[k]['label']}->{gold[k]['label']}" for k in ks if pre[k]["label"] != gold[k]["label"])
    def audit_stats(audit):
        """Claude vs human on one blind sample, and metric ranks against both label sets."""
        ks = sorted(k for k, v in audit.items() if not v.get("bad") and not gold[k]["bad"])
        yc, yh = [pre[k]["label"] for k in ks], [audit[k]["label"] for k in ks]
        cm = confusion_matrix(yc, yh, labels=LABELS)
        res = {
            "n": len(ks), "n_input": len(audit), "acc": float(np.mean([a == b for a, b in zip(yc, yh)])),
            "kappa": float(cohen_kappa_score(yc, yh)),
            "confusion_claude_rows_human_cols": cm.tolist(),
            "human_distribution": Counter(yh), "claude_distribution": Counter(yc),
            # Per Claude label: share the human agreed with (the sample is stratified by Claude label).
            "agree_by_claude_label": {lab: float(cm[i, i] / cm[i].sum()) for i, lab in enumerate(LABELS) if cm[i].sum()},
        }
        # Sensitivity: does the ranking of metrics change when the audited claims are scored
        # against the human labels instead of Claude's?
        sens = {}
        for name, p in preds.items():
            kk = [k for k in ks if k in p]
            sens[name] = {"n": len(kk),
                          "macro_f1_vs_human": macro_f1([audit[k]["label"] for k in kk], [p[k] for k in kk]),
                          "macro_f1_vs_claude": macro_f1([pre[k]["label"] for k in kk], [p[k] for k in kk]),
                          "acc_vs_human": accuracy_score([audit[k]["label"] for k in kk], [p[k] for k in kk])}
        names = [n for n in sens if not (n.startswith("nli_ft") and same_ft)]
        res["sensitivity"] = sens
        res["rank_spearman"] = float(spearmanr([sens[n]["macro_f1_vs_human"] for n in names],
                                                              [sens[n]["macro_f1_vs_claude"] for n in names]).statistic)
        # Human-label GGF per system on the audited claims only.
        res["by_system"] = {
            s: {"n": sum(1 for k in ks if claims[k]["system"] == s),
                "human_supported": float(np.mean([audit[k]["label"] == "supported" for k in ks if claims[k]["system"] == s])),
                "claude_supported": float(np.mean([pre[k]["label"] == "supported" for k in ks if claims[k]["system"] == s]))}
            for s in ("naive", "graph")}
        return res

    audit = load_human("C")
    if audit:
        agreement["audit"] = audit_stats(audit)
    # Quick passes G, H, ...: 30 random claims each, labelled from short excerpts (s6_quick.py). Reported
    # together and pooled with C; pass C alone stays the main audit (its protocol, full passages, is the reference).
    quick, quick_passes = {}, []
    for name in "GHIJ":
        lab = load_human(name)
        if lab:
            quick.update(lab)
            quick_passes.append(name)
    agreement["quick_passes"] = quick_passes
    if quick:
        agreement["audit_G"] = audit_stats(quick)
        if audit:
            agreement["audit_CG"] = audit_stats({**audit, **quick})
    pass_a = load_human("A")
    if pass_a:
        changed = sum(1 for k, v in pass_a.items() if v["label"] != pre[k]["label"])
        agreement["pass_A_excluded"] = {"confirmed": len(pass_a), "labels_changed": changed}
    b = load_human("B")
    if b:
        ks = [k for k in b if k in gold]
        agreement["A_vs_B_kappa"] = cohen_kappa_score([gold[k]["label"] for k in ks], [b[k]["label"] for k in ks])
        agreement["A_vs_B_n"] = len(ks)

    results = {
        "gold": gold_kind, "claims_total": len(claims), "claims_unlabeled": len(unlabeled),
        "claims_bad": len(claims) - len(unlabeled) - len(usable),
        "claims_dev": len(dev), "claims_test": len(test),
        "gold_distribution": Counter(gold[k]["label"] for k in usable),
        "thresholds": thresholds, "ft_identical_to_zs": same_ft, "nosynth_identical_to_zs": same_nosynth,
        "revise": revise,
        "claim_level": claim_table, "comparisons": comparisons, "answer_level_correlation": corr,
        "systems": systems, "qa": qa, "agreement": agreement,
        "judge_coverage": {m: sum(1 for j in judges if j["model"] == m for v in j["judge"].values() if v) for m in ("qwen2.5:3b", "qwen2.5:7b")},
    }
    json.dump(results, open(RES / "results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
    print(json.dumps({k: results[k] for k in ("gold", "claims_test", "gold_distribution", "thresholds", "ft_identical_to_zs")}, ensure_ascii=False, default=float))
    for n, r in sorted(claim_table.items(), key=lambda x: -x[1]["macro_f1"]):
        print(f"{n:22s} macroF1={r['macro_f1']:.3f} [{r['ci95'][0]:.2f},{r['ci95'][1]:.2f}] acc={r['acc']:.3f} n={r['n']}")
    for n, r in corr.items():
        print(f"{n:22s} spearman={r['spearman']:.3f} kendall={r['kendall']:.3f} n={r['n']}")
    print(json.dumps(systems, ensure_ascii=False, default=float)[:600])
    print(json.dumps(qa, ensure_ascii=False))
    print(json.dumps(agreement, ensure_ascii=False, default=str))


if __name__ == "__main__":
    random.seed(SEED)
    main()

In [ ]:
%%writefile /content/vikag-faith/src/s8_revise.py
"""Step 8 (RQ3): verify-then-revise.

For every sampled answer the claim judge (qwen2.5:7b, from s4) flags unsupported/contradicted
claims. The generator (qwen2.5:3b) then rewrites the answer from the SAME retrieved context,
told which claims were not backed by it. The new answer is split (s2 prompt) and judged again
(s4 judge, same evidence pool); at most ROUNDS rounds. Output: outputs/revise.jsonl, one row per
answer with every round, so s7 can compare GGF / contradiction rate / token F1 before and after.
Answers with no flagged claim are kept unchanged (round list empty).
"""

from common import BIG_MODEL, GEN_MODEL, OUT, append_jsonl, chat, chat_json, read_jsonl
from s1_answer import SYSTEM as GEN_SYSTEM, format_context
from s2_claims import FEWSHOT, SYSTEM as SPLIT_SYSTEM, clean
from s4_judges import evidence_text, judge

ROUNDS = 2
REVISE = (
    "Dưới đây là câu trả lời trước của bạn và các mệnh đề trong đó mà người kiểm chứng thấy "
    "KHÔNG được NGỮ CẢNH hỗ trợ hoặc MÂU THUẪN với NGỮ CẢNH. Viết lại câu trả lời: sửa hoặc bỏ "
    "các mệnh đề đó, giữ các ý đúng, chỉ dùng thông tin trong NGỮ CẢNH, 2-5 câu. "
    "Nếu ngữ cảnh không đủ, nói rõ là không tìm thấy thông tin."
)
LABEL_VI = {"unsupported": "không có căn cứ", "contradicted": "mâu thuẫn với ngữ cảnh"}


def split(question, answer):
    prompt = f"{FEWSHOT}\nCÂU HỎI: {question}\nCÂU TRẢ LỜI: {answer}\nJSON:"
    return clean(chat_json(prompt, SPLIT_SYSTEM, GEN_MODEL, num_predict=600).get("claims", []))


def main():
    sample = {r["qid"] for r in read_jsonl(OUT / "sample.jsonl")}
    answers = {(a["qid"], a["system"]): a for a in read_jsonl(OUT / "answers.jsonl")}
    pools = {(p["qid"], p["system"]): p["pool"] for p in read_jsonl(OUT / "pools.jsonl")}
    j7 = {(j["qid"], j["system"]): j["judge"] for j in read_jsonl(OUT / "judges.jsonl")
          if j["model"] == BIG_MODEL}
    claims = {}
    for c in read_jsonl(OUT / "claims.jsonl"):
        claims.setdefault((c["qid"], c["system"]), []).append(c["claim"])
    path = OUT / "revise.jsonl"
    done = {(r["qid"], r["system"]) for r in read_jsonl(path)} if path.exists() else set()
    keys = sorted(k for k in answers if k[0] in sample and k not in done)
    for n, key in enumerate(keys, 1):
        a = answers[key]
        cl = claims.get(key, [])
        labels = [j7[key].get(f"{key[0]}|{key[1]}|{i}") for i in range(len(cl))]
        evidence = evidence_text(pools[key])
        context = format_context(a["retrieved"], a["relations"])
        answer, rounds = a["answer"], []
        for _ in range(ROUNDS):
            bad = [(c, l) for c, l in zip(cl, labels) if l in LABEL_VI]
            if not bad:
                break
            listing = "\n".join(f"- {c} ({LABEL_VI[l]})" for c, l in bad)
            prompt = (f"NGỮ CẢNH:\n{context}\n\nCÂU HỎI: {a['question']}\n\n"
                      f"CÂU TRẢ LỜI TRƯỚC:\n{answer}\n\nMỆNH ĐỀ CẦN SỬA:\n{listing}\n\n{REVISE}")
            answer = chat(prompt, GEN_SYSTEM, GEN_MODEL, num_predict=400, num_ctx=4096)
            cl = split(a["question"], answer)
            labels = judge(cl, evidence, BIG_MODEL) if cl else []
            rounds.append({"answer": answer, "claims": cl, "labels": labels})
        append_jsonl(path, {"qid": key[0], "system": key[1], "rounds": rounds})
        print(f"revise {n}/{len(keys)} rounds={len(rounds)}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/vikag-faith/src/s9_selfcheck.py
"""Step 9: SelfCheckGPT-NLI baseline (reference-free: it never sees the evidence).

  python s9_selfcheck.py gen  # N_SAMPLES extra answers per sampled question, temperature 0.7,
                              # same generator, prompt and context as s1 -> selfcheck_samples.jsonl
  python s9_selfcheck.py nli  # per claim: mean P(entail) / P(contradiction) of the claim given each
                              # sample (mDeBERTa zero-shot) -> selfcheck_scores.jsonl
A claim the stochastic samples do not repeat is treated as likely hallucinated; s7 tunes the
thresholds on dev exactly as for the evidence-based NLI verifier.
"""

import sys

from common import GEN_MODEL, OUT, SEED, USE_CPU, append_jsonl, chat, read_jsonl, write_jsonl

N_SAMPLES = 3
TEMPERATURE = 0.7


def gen():
    from s1_answer import SYSTEM, format_context

    sample = {r["qid"] for r in read_jsonl(OUT / "sample.jsonl")}
    path = OUT / "selfcheck_samples.jsonl"
    done = {(r["qid"], r["system"]) for r in read_jsonl(path)} if path.exists() else set()
    todo = [a for a in read_jsonl(OUT / "answers.jsonl")
            if a["qid"] in sample and (a["qid"], a["system"]) not in done]
    for n, a in enumerate(todo, 1):
        prompt = f"NGỮ CẢNH:\n{format_context(a['retrieved'], a['relations'])}\n\nCÂU HỎI: {a['question']}"
        samples = [chat(prompt, SYSTEM, GEN_MODEL, num_predict=400, num_ctx=4096,
                        temperature=TEMPERATURE, seed=SEED + k) for k in range(1, N_SAMPLES + 1)]
        append_jsonl(path, {"qid": a["qid"], "system": a["system"], "samples": samples})
        print(f"selfcheck gen {n}/{len(todo)}", flush=True)


def nli():
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    from s5_verifier import BASE, MAX_LEN

    dev_ = "cpu" if USE_CPU or not torch.cuda.is_available() else "cuda"
    tok = AutoTokenizer.from_pretrained(BASE)
    model = AutoModelForSequenceClassification.from_pretrained(BASE, torch_dtype=torch.float32).to(dev_).eval()
    samples = {(r["qid"], r["system"]): [s for s in r["samples"] if s.strip()]
               for r in read_jsonl(OUT / "selfcheck_samples.jsonl")}
    rows = []
    claims = [c for c in read_jsonl(OUT / "claims.jsonl") if (c["qid"], c["system"]) in samples]
    for n, c in enumerate(claims, 1):
        prem = samples[(c["qid"], c["system"])]
        if not prem:
            continue
        with torch.no_grad():
            enc = tok(prem, [c["claim"]] * len(prem), truncation="longest_first", max_length=MAX_LEN,
                      padding=True, return_tensors="pt").to(dev_)
            probs = model(**enc).logits.softmax(-1).tolist()
        rows.append({"cid": c["cid"], "entail": sum(p[0] for p in probs) / len(probs),
                     "contra": sum(p[2] for p in probs) / len(probs)})
        if n % 50 == 0:
            print(f"selfcheck nli {n}/{len(claims)}", flush=True)
    write_jsonl(OUT / "selfcheck_scores.jsonl", rows)
    print("selfcheck scores:", len(rows))


if __name__ == "__main__":
    {"gen": gen, "nli": nli}[sys.argv[1]]()

## Cài đặt

In [ ]:
%%bash
set -eo pipefail
DRIVE=/content/drive/MyDrive/vikag-faith
mkdir -p $DRIVE/outputs/logs
cd /content/vikag-faith
[ -L outputs ] || ln -s $DRIVE/outputs outputs
mkdir -p data models
[ -d external/ViHERMES ] || git clone -q --depth 1 https://github.com/ura-hcmut/ViHERMES.git external/ViHERMES
pip install -q -e external/ViHERMES ollama datasets sentencepiece scikit-learn protobuf
for f in sample.jsonl pools.jsonl claims.jsonl answers.jsonl; do test -s outputs/$f || { echo "THIẾU outputs/$f trên Drive"; exit 1; }; done
echo ok

In [ ]:
%%bash
set -eo pipefail
apt-get -qq install -y zstd > /dev/null
command -v ollama > /dev/null || (curl -fsSL https://ollama.com/install.sh | sh > /dev/null)
pgrep -x ollama > /dev/null || (nohup ollama serve > /content/ollama.log 2>&1 &)
sleep 5
cd /content/vikag-faith/external/ViHERMES
[ -f ../../data/graph.db ] || (vihermes extract-contexts --qa dataset/dataset.jsonl --output ../../data/contexts.csv \
  && vihermes ingest --input ../../data/contexts.csv --db ../../data/graph.db)
ls -la ../../data

In [ ]:
import os
os.environ["VIKAG_CPU"] = "0"
os.environ["VIKAG_VINLI"] = "12000"
os.environ["VIKAG_SCORE_FILE"] = "verifier_scores_dot2.jsonl"
os.chdir("/content/vikag-faith/src")

## Phần 1. Triple, mutual index và 20.000 cặp tổng hợp (khoảng 5 phút)

In [ ]:
%%bash
set -eo pipefail
python s3b_triples.py
VIKAG_DATA_TAG=_20k VIKAG_SYNTH_TARGET=20000 python s5_verifier.py data 2>&1 | tail -n 2 | tee ../outputs/logs/dot2_data.log

## Phần 2. mDeBERTa: V3 và fine-tune 20.000 cặp (khoảng 40 phút)

In [ ]:
%%bash
set -eo pipefail
M=../outputs/logs/dot2_mdeberta_done
[ -f $M ] && { echo "đã xong, bỏ qua"; exit 0; }
VIKAG_PRED_BS=32 VIKAG_VARIANTS=v3 VIKAG_SKIP_FT=1 python s5_verifier.py predict 2>&1 | tail -n 2
VIKAG_BS=16 VIKAG_ACCUM=1 VIKAG_EVAL_EVERY=400 VIKAG_DATA_TAG=_20k VIKAG_FT_TAG=ft20 python s5_verifier.py train 2>&1 \
  | grep -E 'transformers|train rows|dev acc|step|best|Error' | tee ../outputs/logs/dot2_train_ft20.log | tail -n 4
VIKAG_PRED_BS=32 VIKAG_VARIANTS=v1,v3 VIKAG_SKIP_ZS=1 VIKAG_FT_TAG=ft20 python s5_verifier.py predict 2>&1 | tail -n 3
touch $M

## Phần 3. LLM-judge và RAGAS với Qwen2.5-14B (khoảng 45 phút)

In [ ]:
%%bash
set -eo pipefail
M=../outputs/logs/dot2_judge14_done
[ -f $M ] && { echo "đã xong, bỏ qua"; exit 0; }
ollama pull qwen2.5:14b > /dev/null 2>&1
VIKAG_JUDGE_FILE=judges_dot2.jsonl python s4_judges.py qwen2.5:14b 2>&1 | tail -n 2
ollama stop qwen2.5:14b || true
touch $M

## Phần 4. XLM-R: zero-shot và fine-tune (khoảng 30 phút)

In [ ]:
%%bash
set -eo pipefail
M=../outputs/logs/dot2_xlmr_done
[ -f $M ] && { echo "đã xong, bỏ qua"; exit 0; }
export VIKAG_BASE=joeddav/xlm-roberta-large-xnli VIKAG_ZS_NAME=xzs VIKAG_FT_TAG=xft
VIKAG_PRED_BS=16 VIKAG_VARIANTS=v1,v3 VIKAG_SKIP_FT=1 python s5_verifier.py predict 2>&1 | tail -n 2
VIKAG_BS=8 VIKAG_ACCUM=2 VIKAG_EVAL_EVERY=400 VIKAG_DATA_TAG=_20k VIKAG_MAX_RUN=16000 python s5_verifier.py train 2>&1 \
  | grep -E 'transformers|train rows|dev acc|step|best|Error' | tee ../outputs/logs/dot2_train_xft.log | tail -n 4
VIKAG_PRED_BS=16 VIKAG_VARIANTS=v1,v3 VIKAG_SKIP_ZS=1 python s5_verifier.py predict 2>&1 | tail -n 3
touch $M

## Phần 5. Verifier Qwen nhỏ trả nhãn và span (khoảng 20 phút)

In [ ]:
%%bash
set -eo pipefail
M=../outputs/logs/dot2_span_done
[ -f $M ] && { echo "đã xong, bỏ qua"; exit 0; }
ollama pull qwen2.5:3b > /dev/null 2>&1
VIKAG_SPAN=1 python s4_judges.py qwen2.5:3b 2>&1 | tail -n 2
touch $M

## Đóng gói

In [ ]:
%%bash
cd /content/drive/MyDrive/vikag-faith
rm -f vikag_dot2.zip
zip -qr vikag_dot2.zip outputs/verifier_scores_dot2.jsonl outputs/judges_dot2.jsonl outputs/judges_span.jsonl outputs/triple_index.json outputs/logs
ls -la vikag_dot2.zip
ls outputs/logs | grep dot2